In [ ]:

# Setup

import numpy as np
import pandas as pd
from pathlib import Path
import os

import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker

OUTPUT = Path()

# Skripte ausführen 

%run "Data Prep.py"
%run "Factor Calc.py"

['Date', 'EEX-EU CO2 Emissions', 'STOXX 600 Europe', 'RF_EUR', 'Mkt-RF_EUR', '5 SMB_EUR', '5 HML_EUR', '5 RMW_EUR', '5 CMA_EUR', 'WML_EUR', 'ISIN', 'Price Close', 'Monthly_Returns', 'Offset/Credit', 'Scope1', 'Scope2', 'Company Name', 'RIC', 'Industry', 'Company Market Cap', 'Country of Headquarters', 'Revenue', 'Excess Return', 'Scope1_Intensity', 'Scope2_Intensity', 'OC_Intensity', 'Scope12', 'Scope12_Intensity', 'Scope_Total', 'Scope_Total_Intensity', 'ScopeTotal_Changes', 'Scope12_Changes']


In [116]:
###########################################################################
# Tabelle 1 – Deskriptive Statistik: Carbon Intensity
###########################################################################

emission_var = 'Scope_Total_Intensity'
industry_col = 'Industry'

# Eine Beobachtung pro Unternehmen (letztes verfügbares Jahr)
df_desc = (
    df.sort_values(['ISIN', 'Date'])
      .groupby('ISIN', as_index=False)
      .last()
    [[industry_col, 'ISIN', emission_var, 'Scope1', 'Scope2', 'Offset/Credit', 'Company Market Cap', 'Revenue']]
      .dropna(subset=[emission_var, industry_col])
)

# Statistiken pro Industrie
desc_table = (
    df_desc.groupby(industry_col)
    .agg(
        N=('ISIN', 'count'),
        Scope1=('Scope1', 'mean'),
        Scope2=('Scope2', 'mean'),
        Offset_Credit=('Offset/Credit', 'mean'),
        Mean=(emission_var, 'mean'),
        Median=(emission_var, 'median'),
        Q25=(emission_var, lambda x: x.quantile(0.25)),
        Q75=(emission_var, lambda x: x.quantile(0.75)),
        MCap_Mean      = ('Company Market Cap',  'mean'),   
        MCap_Median    = ('Company Market Cap',  'median'),
        Revenue_Mean   = ('Revenue',            'mean'),   
        Revenue_Median = ('Revenue',            'median'),
    )
    .round(2)
    .sort_values('Scope1', ascending=False)
    .reset_index()
)

# Gesamtzeile anhängen
total_row = pd.DataFrame([{
    'Industry': 'Total',
    'N':        df_desc['ISIN'].nunique(),
    'Scope1': df_desc['Scope1'].mean().round(0),
    'Scope2': df_desc['Scope2'].mean().round(0),
    'Offset_Credit': df_desc['Offset/Credit'].mean().round(0),
    'Mean':     df_desc[emission_var].mean().round(2),
    'Median':   df_desc[emission_var].median().round(2),
    'Q25':      df_desc[emission_var].quantile(0.25).round(2),
    'Q75':      df_desc[emission_var].quantile(0.75).round(2),
    'MCap_Mean':    df_desc['Company Market Cap'].mean().round(2),    
    'MCap_Median':  df_desc['Company Market Cap'].median().round(2),
    'Revenue_Mean':  df_desc['Revenue'].mean().round(2),   
    'Revenue_Median':df_desc['Revenue'].median().round(2),
}])

desc_table = pd.concat([desc_table, total_row], ignore_index=True)

In [117]:
###########################################################################
# Tabelle: Datenverfügbarkeit Emissionsdaten pro Jahr
###########################################################################

df_coverage = df.copy()
df_coverage['Date'] = to_datetime_safe(df_coverage['Date'])
df_coverage['Year'] = df_coverage['Date'].dt.year

# Eine Beobachtung pro ISIN-Jahr
df_annual = (
    df_coverage.sort_values(['ISIN', 'Date'])
               .groupby(['ISIN', 'Year'], as_index=False)
               .last()
    [['ISIN', 'Year', 'Scope1', 'Scope2', 'Offset/Credit']]
)

# Anzahl Unternehmen mit Daten pro Jahr
coverage_table = (
    df_annual.groupby('Year')
    .agg(
        N_Total         = ('ISIN',          'count'),
        N_Scope1        = ('Scope1',        lambda x: (x > 0).sum()),
        N_Scope2        = ('Scope2',        lambda x: (x > 0).sum()),
        N_Offset_Credit = ('Offset/Credit', lambda x: (x > 0).sum()),
    )
    .reset_index()
)

In [118]:
###########################################################################
# Carbon Emissionen over time
###########################################################################

emission_var_plot = 'Scope12_Intensity'
emission_abs_plot = 'Scope12'
n                 = 5

df_plot = df.copy()
df_plot['Date'] = to_datetime_safe(df_plot['Date'])
df_plot['Year'] = df_plot['Date'].dt.year

annual = (
    df_plot[['ISIN', 'Year', emission_var]]
    .drop_duplicates(subset=['ISIN', 'Year'])
    .dropna(subset=[emission_var])
    .copy()
)

annual['Bucket'] = annual.groupby('Year')[emission_var].transform(
    lambda x: safe_quantile_sort(x, n)
)
annual = annual.dropna(subset=['Bucket'])
annual['Bucket']      = annual['Bucket'].astype('Int64')
annual['valid_from']  = pd.to_datetime((annual['Year'] + 1).astype(str) + '-07-01')
annual['valid_until'] = pd.to_datetime((annual['Year'] + 2).astype(str) + '-06-30')

df_dates    = df_plot[['ISIN', 'Date']].drop_duplicates()
assignments = df_dates.merge(
    annual[['ISIN', 'valid_from', 'valid_until', 'Bucket']], on='ISIN', how='left'
)
assignments = assignments[
    (assignments['Date'] >= assignments['valid_from']) &
    (assignments['Date'] <= assignments['valid_until'])
]

df_sorted_plot = (
    df_plot
    .merge(
        annual[['ISIN', 'Year', 'Bucket']],
        on=['ISIN', 'Year'],
        how='left'
    )
    .dropna(subset=['Bucket'])
)

df_q1_q5 = df_sorted_plot[df_sorted_plot['Bucket'].isin([1, 5])].copy()
df_q1_q5['Year'] = df_q1_q5['Date'].dt.year

annual_avg = (
    df_q1_q5.groupby(['Year', 'Bucket'])[[emission_abs_plot, emission_var_plot, 'Scope12_Changes']]
    .mean()
    .reset_index()
)

q1 = annual_avg[annual_avg['Bucket'] == 1].set_index('Year')
q5 = annual_avg[annual_avg['Bucket'] == 5].set_index('Year')

def build_export_table(q1, q5, col):
    return pd.DataFrame({
        'Green (Q1)': q1[col],
        'Brown (Q5)': q5[col],
    })

ts_absolute = build_export_table(q1, q5, emission_abs_plot)
ts_intensity = build_export_table(q1, q5, emission_var_plot)
ts_changes   = build_export_table(q1, q5, 'Scope12_Changes')

In [119]:
###########################################################################
# Quintil Performance
###########################################################################

# Total Return Portfolios (ohne RF-Abzug) für Marktvergleich
def build_quantile_portfolios_total_return(df, emission_var, n, weighting='value'):
   
    df = df.copy()
    df['Date'] = df['Date'].dt.to_timestamp() if isinstance(df['Date'].dtype, pd.PeriodDtype) else pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year

    annual = (
        df[['ISIN', 'Year', emission_var]]
        .drop_duplicates(subset=['ISIN', 'Year'])
        .dropna(subset=[emission_var])
        .copy()
    )
    annual['Bucket'] = annual.groupby('Year')[emission_var].transform(
        lambda x: safe_quantile_sort(x, n)
    )
    annual = annual.dropna(subset=['Bucket'])
    annual['Bucket']      = annual['Bucket'].astype('Int64')
    annual['valid_from']  = pd.to_datetime((annual['Year'] + 1).astype(str) + '-07-01')
    annual['valid_until'] = pd.to_datetime((annual['Year'] + 2).astype(str) + '-06-30')

    df_dates    = df[['ISIN', 'Date']].drop_duplicates()
    assignments = df_dates.merge(
        annual[['ISIN', 'valid_from', 'valid_until', 'Bucket']], on='ISIN', how='left'
    )
    assignments = assignments[
        (assignments['Date'] >= assignments['valid_from']) &
        (assignments['Date'] <= assignments['valid_until'])
    ]

    df_sorted = (
        df.merge(assignments[['ISIN', 'Date', 'Bucket']], on=['ISIN', 'Date'], how='left')
          .dropna(subset=['Bucket', 'Monthly_Returns'])
    )

    if weighting == 'value':
        df_sorted = df_sorted.dropna(subset=['Company Market Cap']).copy()
        df_sorted['_vw'] = df_sorted['Monthly_Returns'] * df_sorted['Company Market Cap']
        grp = df_sorted.groupby(['Date', 'Bucket'], observed=True)
        port_returns = (grp['_vw'].sum() / grp['Company Market Cap'].sum()).unstack('Bucket')
    else:
        port_returns = (
            df_sorted.groupby(['Date', 'Bucket'], observed=True)['Monthly_Returns']
            .mean()
            .unstack('Bucket')
        )

    port_returns.columns = [f'P{int(c)}' for c in port_returns.columns]
    port_returns['LS']   = port_returns[f'P{n}'] - port_returns['P1']
    return port_returns.astype(float)


# Total Return Portfolios für Marktvergleich
port_total_q5      = build_quantile_portfolios_total_return(df, emission_var, n=5, weighting='value')
port_total_terzile = build_quantile_portfolios_total_return(df, emission_var, n=3, weighting='value')

# Marktadjustiert: Total Return Portfolio minus STOXX 600 Raw Return
market_adj         = calc_market_adj(port_total_q5,      stoxx_returns)
market_adj_terzile = calc_market_adj(port_total_terzile, stoxx_returns)

# Kumulierte Returns
cumulative_returns             = normalize_to_100(100 * (1 + port_ew).cumprod())           
cumulative_market_adj          = normalize_to_100(100 * (1 + market_adj).cumprod())         
cumulative_returns_terzile     = normalize_to_100(100 * (1 + port_terzile).cumprod())       
cumulative_market_adj_terzile  = normalize_to_100(100 * (1 + market_adj_terzile).cumprod()) 

###########################################################################
# Performance-Tabelle
###########################################################################

perf_rows = {}

for col in port_ew.columns:
    ret = port_ew[col].dropna()
    rf  = rf_ts.reindex(ret.index).fillna(0)

    mean_ret    = ret.mean()
    std_ret     = ret.std()
    n_obs       = len(ret)
    t_stat      = mean_ret / (std_ret / np.sqrt(n_obs))
    p_value     = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_obs - 1))   # zweiseitig
    sharpe      = (ret - rf).mean() / std_ret * np.sqrt(12)

    # Signifikanzsterne
    if p_value < 0.01:
        stars = '***'
    elif p_value < 0.05:
        stars = '**'
    elif p_value < 0.10:
        stars = '*'
    else:
        stars = ''

    perf_rows[col] = {
        'Excess Return (%)': round(mean_ret * 100, 4),
        'Std. Dev. (%)':     round(std_ret  * 100, 4),
        't-Statistic':            round(t_stat, 3),
        'p-Value':                round(p_value, 3),
        'Significance':           stars,
        'Sharpe Ratio (ann.)':    round(sharpe, 3),
    }

perf_table = pd.DataFrame(perf_rows).T
# print(perf_table.to_string())

In [120]:
###########################################################################
# Industrie Portfolio Performance
###########################################################################

# Anzahl Unternehmen pro Industrie (aus df_desc bereits berechnet)
n_companies = (
    df.dropna(subset=['Industry'])
      .groupby('Industry')['ISIN']
      .nunique()
      .rename('N Companies')
)

def normalize_to_100(cum_returns):
    """Setzt den ersten Wert auf 100 durch Voranstellen einer Startzeile."""
    start = pd.DataFrame(
        100,
        index   = [cum_returns.index[0] - pd.DateOffset(months=1)],
        columns = cum_returns.columns
    )
    return pd.concat([start, cum_returns])

# industry_returns bereits gebaut (VW) – direkt weiterverwenden
# Kumulierte Returns pro Industrie
cumulative_industry = normalize_to_100(100 * (1 + industry_returns).cumprod())

# Performance-Tabelle pro Industrie
rf_ts = (
    df[['Date', 'RF_EUR']].drop_duplicates('Date').copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .set_index('Date')['RF_EUR']
)

perf_industry = {}

for sector in industry_returns.columns:
    ret = industry_returns[sector].dropna()
    rf  = rf_ts.reindex(ret.index).fillna(0)

    mean_ret = ret.mean()
    std_ret  = ret.std()
    n_obs    = len(ret)
    t_stat   = mean_ret / (std_ret / np.sqrt(n_obs))
    p_value  = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_obs - 1))
    sharpe   = (ret - rf).mean() / std_ret * np.sqrt(12)

    if p_value < 0.01:
        stars = '***'
    elif p_value < 0.05:
        stars = '**'
    elif p_value < 0.10:
        stars = '*'
    else:
        stars = ''

    perf_industry[sector] = {
        'Excess Return (%)':     round(mean_ret * 100, 4),
        'Std. Dev. (%)':         round(std_ret  * 100, 4),
        't-Statistic':           round(t_stat, 3),
        'p-Value':               round(p_value, 3),
        'Significance':          stars,
        'Sharpe Ratio (ann.)':   round(sharpe, 3),
    }

perf_industry_df = (
    pd.DataFrame(perf_industry).T
      .sort_values('Excess Return (%)', ascending=False)
)

In [121]:
###########################################################################
# Faktoren Korrelationstabelle
###########################################################################

# Alle Faktoren zusammenführen
factors_corr = (
    df[['Date'] + factors].drop_duplicates('Date').copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .set_index('Date')
      .join(co2_factor)
      .join(all_bmg_factors['Scope12_Intensity'].rename('BMG'))
      .dropna()
)

factor_labels = {
    'Mkt-RF_EUR': 'Mkt-RF',
    '5 SMB_EUR':  'SMB',
    '5 HML_EUR':  'HML',
    '5 RMW_EUR':  'RMW',
    '5 CMA_EUR':  'CMA',
    'WML_EUR':    'MOM',
    'CO2_Ret':    'CO2',
    'BMG':        'BMG',
}

factors_corr = factors_corr.rename(columns=factor_labels)
factor_cols  = list(factor_labels.values())

# --- Zusammenfassungsstatistiken pro Faktor ---
summary_rows = {}

for col in factor_cols:
    ret     = factors_corr[col].dropna()
    n_obs   = len(ret)
    mean    = ret.mean() * 100          # in %
    sigma   = ret.std()  * 100          # in %
    t_stat  = ret.mean() / (ret.std() / np.sqrt(n_obs))
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_obs - 1))

    if p_value < 0.01:   stars = '***'
    elif p_value < 0.05: stars = '**'
    elif p_value < 0.10: stars = '*'
    else:                stars = ''

    summary_rows[col] = {
        'Mean (%)':    round(mean, 3),
        'Sig.':        stars,
        't-Statistic': round(t_stat, 3),
        'Sigma (%)':   round(sigma, 3),
    }

summary_df = pd.DataFrame(summary_rows).T

# --- Korrelationsmatrix ---
corr_matrix = factors_corr[factor_cols].corr().round(3)

# --- Zusammenführen: Summary + Korrelation ---
corr_table = summary_df.join(corr_matrix)

In [122]:
###########################################################################
# BMG – Vergleich
###########################################################################

bmg_labels = {
    'Scope_Total_Intensity': 'BMG (Total Intensity)',
    'Scope12_Intensity':     'BMG (Scope12 Intensity)',
    'Scope_Total':           'BMG (Total Absolute)',
    'Scope12':               'BMG (Scope12 Absolute)',
    'ScopeTotal_Changes':    'BMG (Total Changes)',
    'Scope12_Changes':       'BMG (Scope12 Changes)',
}

bmg_df = pd.DataFrame({
    bmg_labels[k]: v for k, v in all_bmg_factors.items()
}).dropna()

# --- 1. Deskriptive Statistik ---
bmg_stats = {}

for col in bmg_df.columns:
    ret     = bmg_df[col]
    n_obs   = len(ret)
    mean    = ret.mean() * 100
    sigma   = ret.std()  * 100
    t_stat  = ret.mean() / (ret.std() / np.sqrt(n_obs))
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_obs - 1))
    sharpe  = ret.mean() / ret.std() * np.sqrt(12)

    if p_value < 0.01:   stars = '***'
    elif p_value < 0.05: stars = '**'
    elif p_value < 0.10: stars = '*'
    else:                stars = ''

    bmg_stats[col] = {
        'Mean (%)':           round(mean, 3),
        'Sig.':               stars,
        't-Statistic':        round(t_stat, 3),
        'Std. Dev. (%)':      round(sigma, 3),
        'Sharpe (ann.)':      round(sharpe, 3),
        'N':                  n_obs,
    }

bmg_stats_df = pd.DataFrame(bmg_stats).T

# --- 2. Korrelationstabelle ---
bmg_corr = bmg_df.corr().round(3)

# Gruppe kennzeichnen: Intensity / Absolute / Changes
bmg_groups = pd.Series({
    'BMG (Total Intensity)':  'Intensity',
    'BMG (Scope12 Intensity)':'Intensity',
    'BMG (Total Absolute)':   'Absolute',
    'BMG (Scope12 Absolute)': 'Absolute',
    'BMG (Total Changes)':    'Changes',
    'BMG (Scope12 Changes)':  'Changes',
}, name='Group')

bmg_stats_df.insert(0, 'Group', bmg_groups)

def normalize_to_100(cum_returns):
    """Setzt den ersten Wert auf 100 durch Voranstellen einer Startzeile."""
    start = pd.DataFrame(
        100,
        index   = [cum_returns.index[0] - pd.DateOffset(months=1)],
        columns = cum_returns.columns
    )
    return pd.concat([start, cum_returns])

# --- 3. Kumulative Returns ---
bmg_cumulative = normalize_to_100(100 * (1 + bmg_df).cumprod() - 1)

In [123]:
###########################################################################
# Regressionstabelle - Qunitil, Terzile, Branchen - equal weighted
###########################################################################

def stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.3f}{stars(p)}'

def build_regression_table(port_returns, reg_data, factors, median_intensity=None):
 
    lags             = int(len(reg_data) ** (1 / 3))
    factors_with_bmg = factors + ['BMG']
    rows             = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_bmg  = sm.add_constant(reg_data[factors_with_bmg].astype(float))
        m_bmg  = sm.OLS(y, X_bmg).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_bmg, y, X_base, X_bmg)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_bmg.params['const'],      m_bmg.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_bmg.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_bmg.params['5 SMB_EUR'],  m_bmg.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_bmg.params['5 HML_EUR'],  m_bmg.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_bmg.params['5 RMW_EUR'],  m_bmg.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_bmg.params['5 CMA_EUR'],  m_bmg.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_bmg.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            'BMG':        fmt_coef(m_bmg.params['BMG'],        m_bmg.pvalues['BMG']),
            'Adj. R² (%)':       f'{m_bmg.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_bmg.params['const']      - m_base.params['const'],      m_bmg.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_bmg.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_bmg.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_bmg.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_bmg.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_bmg.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_bmg.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_bmg.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_bmg.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_bmg.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_bmg.params['WML_EUR']    - m_base.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_bmg.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Faktoren-Zeitreihe (gemeinsam für alle Tabellen)
###########################################################################

factors_ts = (
    df[['Date'] + factors].drop_duplicates('Date').copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .set_index('Date')
)

bmg_series = all_bmg_factors['Scope12_Intensity']

###########################################################################
# Tabelle 1 – Quintile EW
###########################################################################

reg_data_q5 = (
    quantile_portfolios['Quintile_EW']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Quintil
median_q5 = (
    df_sorted_plot.dropna(subset=['Bucket', 'Scope12_Intensity'])
                  .groupby('Bucket')['Scope12_Intensity']
                  .median().round(3)
)
median_q5.index = [f'P{int(i)}' for i in median_q5.index]
median_q5['LS'] = np.nan

table_quintile = build_regression_table(
    port_returns      = quantile_portfolios['Quintile_EW'],
    reg_data          = reg_data_q5,
    factors           = factors,
    median_intensity  = median_q5.to_dict(),
)
table_quintile.index.name = 'Quintile'

###########################################################################
# Tabelle 2 – Terzile EW
###########################################################################

reg_data_t3 = (
    quantile_portfolios['Terzile_EW']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Terzil
df_sorted_t3 = build_quantile_portfolios(df, emission_var, n=3, weighting='equal')[1] if False else \
    df_sorted_plot.copy()   

annual_t3 = (
    df[['ISIN', 'Date', emission_var]].copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .assign(Year=lambda x: x['Date'].dt.year)
      .drop_duplicates(subset=['ISIN', 'Year'])
      .dropna(subset=[emission_var])
)
annual_t3['Bucket'] = annual_t3.groupby('Year')[emission_var].transform(
    lambda x: safe_quantile_sort(x, 3)
)

median_t3 = (
    annual_t3.dropna(subset=['Bucket'])
             .groupby('Bucket')[emission_var]
             .median().round(3)
)
median_t3.index = [f'P{int(i)}' for i in median_t3.index]
median_t3['LS'] = np.nan

table_terzile = build_regression_table(
    port_returns      = quantile_portfolios['Terzile_EW'],
    reg_data          = reg_data_t3,
    factors           = factors,
    median_intensity  = median_t3.to_dict(),
)
table_terzile.index.name = 'Terzile'

###########################################################################
# Tabelle 3 – Branchen
###########################################################################

industry_returns_ew = build_industry_portfolios(df, industry_col='Industry', weighting='equal')

reg_data_ind = (
    industry_returns_ew
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Branche
median_industry = (
    df.dropna(subset=['Industry', 'Scope12_Intensity'])
      .groupby('Industry')['Scope12_Intensity']
      .median().round(3)
      .to_dict()
)

table_industry = build_regression_table(
    port_returns      = industry_returns_ew,
    reg_data          = reg_data_ind,
    factors           = factors,
    median_intensity  = median_industry,
)
table_industry.index.name = 'Industry'
table_industry = table_industry.sort_values('BMG', ascending=False)

In [124]:
###########################################################################
# Regressionstabelle - Qunitil, Terzile, Branchen - value weightes
###########################################################################

def stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.3f}{stars(p)}'

def build_regression_table(port_returns, reg_data, factors, median_intensity=None):
 
    lags             = int(len(reg_data) ** (1 / 3))
    factors_with_bmg = factors + ['BMG']
    rows             = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_bmg  = sm.add_constant(reg_data[factors_with_bmg].astype(float))
        m_bmg  = sm.OLS(y, X_bmg).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_bmg, y, X_base, X_bmg)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_bmg.params['const'],      m_bmg.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_bmg.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_bmg.params['5 SMB_EUR'],  m_bmg.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_bmg.params['5 HML_EUR'],  m_bmg.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_bmg.params['5 RMW_EUR'],  m_bmg.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_bmg.params['5 CMA_EUR'],  m_bmg.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_bmg.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            'BMG':        fmt_coef(m_bmg.params['BMG'],        m_bmg.pvalues['BMG']),
            'Adj. R² (%)':       f'{m_bmg.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_bmg.params['const']      - m_base.params['const'],      m_bmg.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_bmg.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_bmg.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_bmg.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_bmg.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_bmg.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_bmg.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_bmg.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_bmg.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_bmg.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_bmg.params['WML_EUR']    - m_base.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_bmg.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Faktoren-Zeitreihe (gemeinsam für alle Tabellen)
###########################################################################

factors_ts = (
    df[['Date'] + factors].drop_duplicates('Date').copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .set_index('Date')
)

bmg_series = all_bmg_factors['Scope12_Intensity']

###########################################################################
# Tabelle 1 – Quintile EW
###########################################################################

reg_data_q5 = (
    quantile_portfolios['Quintile_VW']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Quintil
median_q5 = (
    df_sorted_plot.dropna(subset=['Bucket', 'Scope12_Intensity'])
                  .groupby('Bucket')['Scope12_Intensity']
                  .median().round(3)
)
median_q5.index = [f'P{int(i)}' for i in median_q5.index]
median_q5['LS'] = np.nan

table_vw_quintile = build_regression_table(
    port_returns      = quantile_portfolios['Quintile_VW'],
    reg_data          = reg_data_q5,
    factors           = factors,
    median_intensity  = median_q5.to_dict(),
)
table_vw_quintile.index.name = 'Quintile_vw'

###########################################################################
# Tabelle 2 – Terzile EW
###########################################################################

reg_data_t3 = (
    quantile_portfolios['Terzile_VW']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Terzil
df_sorted_t3 = build_quantile_portfolios(df, emission_var, n=3, weighting='equal')[1] if False else \
    df_sorted_plot.copy()   

annual_t3 = (
    df[['ISIN', 'Date', emission_var]].copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .assign(Year=lambda x: x['Date'].dt.year)
      .drop_duplicates(subset=['ISIN', 'Year'])
      .dropna(subset=[emission_var])
)
annual_t3['Bucket'] = annual_t3.groupby('Year')[emission_var].transform(
    lambda x: safe_quantile_sort(x, 3)
)

median_t3 = (
    annual_t3.dropna(subset=['Bucket'])
             .groupby('Bucket')[emission_var]
             .median().round(3)
)
median_t3.index = [f'P{int(i)}' for i in median_t3.index]
median_t3['LS'] = np.nan

table_vw_terzile = build_regression_table(
    port_returns      = quantile_portfolios['Terzile_VW'],
    reg_data          = reg_data_t3,
    factors           = factors,
    median_intensity  = median_t3.to_dict(),
)
table_vw_terzile.index.name = 'Terzile_vw'

###########################################################################
# Tabelle 3 – Branchen
###########################################################################

industry_returns_vw = build_industry_portfolios(df, industry_col='Industry', weighting='value')

reg_data_ind = (
    industry_returns_vw
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Branche
median_industry = (
    df.dropna(subset=['Industry', 'Scope12_Intensity'])
      .groupby('Industry')['Scope12_Intensity']
      .median().round(3)
      .to_dict()
)

table_vw_industry = build_regression_table(
    port_returns      = industry_returns_vw,
    reg_data          = reg_data_ind,
    factors           = factors,
    median_intensity  = median_industry,
)
table_vw_industry.index.name = 'Industry_vw'
table_vw_industry = table_vw_industry.sort_values('BMG', ascending=False)

In [125]:
###########################################################################
# Regressionstabelle - Qunitil, Terzile, Branchen - Emission weightes
###########################################################################

def stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.3f}{stars(p)}'

def build_regression_table(port_returns, reg_data, factors, median_intensity=None):
 
    lags             = int(len(reg_data) ** (1 / 3))
    factors_with_bmg = factors + ['BMG']
    rows             = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_bmg  = sm.add_constant(reg_data[factors_with_bmg].astype(float))
        m_bmg  = sm.OLS(y, X_bmg).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_bmg, y, X_base, X_bmg)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_bmg.params['const'],      m_bmg.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_bmg.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_bmg.params['5 SMB_EUR'],  m_bmg.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_bmg.params['5 HML_EUR'],  m_bmg.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_bmg.params['5 RMW_EUR'],  m_bmg.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_bmg.params['5 CMA_EUR'],  m_bmg.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_bmg.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            'BMG':        fmt_coef(m_bmg.params['BMG'],        m_bmg.pvalues['BMG']),
            'Adj. R² (%)':       f'{m_bmg.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_bmg.params['const']      - m_base.params['const'],      m_bmg.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_bmg.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_bmg.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_bmg.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_bmg.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_bmg.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_bmg.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_bmg.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_bmg.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_bmg.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_bmg.params['WML_EUR']    - m_base.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_bmg.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Faktoren-Zeitreihe (gemeinsam für alle Tabellen)
###########################################################################

factors_ts = (
    df[['Date'] + factors].drop_duplicates('Date').copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .set_index('Date')
)

bmg_series = all_bmg_factors['Scope12_Intensity']

###########################################################################
# Tabelle 1 – Quintile EW
###########################################################################

reg_data_q5 = (
    quantile_portfolios['Quintile_EW_Emission']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Quintil
median_q5 = (
    df_sorted_plot.dropna(subset=['Bucket', 'Scope12_Intensity'])
                  .groupby('Bucket')['Scope12_Intensity']
                  .median().round(3)
)
median_q5.index = [f'P{int(i)}' for i in median_q5.index]
median_q5['LS'] = np.nan

table_ewe_quintile = build_regression_table(
    port_returns      = quantile_portfolios['Quintile_EW_Emission'],
    reg_data          = reg_data_q5,
    factors           = factors,
    median_intensity  = median_q5.to_dict(),
)
table_ewe_quintile.index.name = 'Quintile_EW_Emission'

###########################################################################
# Tabelle 2 – Terzile EW
###########################################################################

reg_data_t3 = (
    quantile_portfolios['Terzile_EW_Emission']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Terzil
df_sorted_t3 = build_quantile_portfolios(df, emission_var, n=3, weighting='equal')[1] if False else \
    df_sorted_plot.copy()   

annual_t3 = (
    df[['ISIN', 'Date', emission_var]].copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .assign(Year=lambda x: x['Date'].dt.year)
      .drop_duplicates(subset=['ISIN', 'Year'])
      .dropna(subset=[emission_var])
)
annual_t3['Bucket'] = annual_t3.groupby('Year')[emission_var].transform(
    lambda x: safe_quantile_sort(x, 3)
)

median_t3 = (
    annual_t3.dropna(subset=['Bucket'])
             .groupby('Bucket')[emission_var]
             .median().round(3)
)
median_t3.index = [f'P{int(i)}' for i in median_t3.index]
median_t3['LS'] = np.nan

table_ewe_terzile = build_regression_table(
    port_returns      = quantile_portfolios['Terzile_EW_Emission'],
    reg_data          = reg_data_t3,
    factors           = factors,
    median_intensity  = median_t3.to_dict(),
)
table_ewe_terzile.index.name = 'Terzile_EW_Emission'

###########################################################################
# Tabelle 3 – Branchen
###########################################################################

industry_returns_ewe = build_industry_portfolios(df, industry_col='Industry', weighting='ew_emission')

reg_data_ind = (
    industry_returns_ewe
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Branche
median_industry = (
    df.dropna(subset=['Industry', 'Scope12_Intensity'])
      .groupby('Industry')['Scope12_Intensity']
      .median().round(3)
      .to_dict()
)

table_ewe_industry = build_regression_table(
    port_returns      = industry_returns_ewe,
    reg_data          = reg_data_ind,
    factors           = factors,
    median_intensity  = median_industry,
)
table_ewe_industry.index.name = 'Industry_EW_Emission'
table_ewe_industry = table_ewe_industry.sort_values('BMG', ascending=False)

In [126]:
###########################################################################
# Regressionstabelle – Makrofaktor
###########################################################################

def build_co2_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags              = int(len(reg_data) ** (1 / 3))
    factors_with_co2  = factors + ['CO2_Ret']
    rows              = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_co2  = sm.add_constant(reg_data[factors_with_co2].astype(float))
        m_co2  = sm.OLS(y, X_co2).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_co2, y, X_base, X_co2)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_co2.params['const'],      m_co2.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_co2.params['Mkt-RF_EUR'], m_co2.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_co2.params['5 SMB_EUR'],  m_co2.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_co2.params['5 HML_EUR'],  m_co2.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_co2.params['5 RMW_EUR'],  m_co2.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_co2.params['5 CMA_EUR'],  m_co2.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_co2.params['WML_EUR'],    m_co2.pvalues['WML_EUR']),
            'CO2':               fmt_coef(m_co2.params.get('CO2_Ret', np.nan), m_co2.pvalues.get('CO2_Ret', 1)),
            'Adj. R² (%)':       f'{m_co2.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_co2.params['const']      - m_base.params['const'],      m_co2.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_co2.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_co2.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_co2.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_co2.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_co2.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_co2.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_co2.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_co2.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_co2.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_co2.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_co2.params['WML_EUR']    - m_base.params['WML_EUR'],    m_co2.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_co2.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + CO2 joinen
###########################################################################

factors_ts_co2 = factors_ts.join(co2_factor)

# Quintile
reg_data_q5_co2 = (
    quantile_portfolios['Quintile_EW']
    .join(factors_ts_co2)
    .dropna()
)

table_co2_quintile = build_co2_regression_table(
    port_returns     = quantile_portfolios['Quintile_EW'],
    reg_data         = reg_data_q5_co2,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_co2_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_co2 = (
    quantile_portfolios['Terzile_EW']
    .join(factors_ts_co2)
    .dropna()
)

table_co2_terzile = build_co2_regression_table(
    port_returns     = quantile_portfolios['Terzile_EW'],
    reg_data         = reg_data_t3_co2,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_co2_terzile.index.name = 'Terzile'

# Branchen
reg_data_ind_co2 = (
    industry_returns
    .join(factors_ts_co2)
    .dropna()
)

table_co2_industry = build_co2_regression_table(
    port_returns     = industry_returns,
    reg_data         = reg_data_ind_co2,
    factors          = factors,
    median_intensity = median_industry,
)
table_co2_industry.index.name = 'Industry'
table_co2_industry = table_co2_industry.sort_values('CO2', ascending=False)

In [127]:
###########################################################################
# Regressionstabelle – Makrofaktor - value weighted
###########################################################################

def build_co2_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags              = int(len(reg_data) ** (1 / 3))
    factors_with_co2  = factors + ['CO2_Ret']
    rows              = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_co2  = sm.add_constant(reg_data[factors_with_co2].astype(float))
        m_co2  = sm.OLS(y, X_co2).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_co2, y, X_base, X_co2)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_co2.params['const'],      m_co2.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_co2.params['Mkt-RF_EUR'], m_co2.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_co2.params['5 SMB_EUR'],  m_co2.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_co2.params['5 HML_EUR'],  m_co2.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_co2.params['5 RMW_EUR'],  m_co2.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_co2.params['5 CMA_EUR'],  m_co2.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_co2.params['WML_EUR'],    m_co2.pvalues['WML_EUR']),
            'CO2':               fmt_coef(m_co2.params.get('CO2_Ret', np.nan), m_co2.pvalues.get('CO2_Ret', 1)),
            'Adj. R² (%)':       f'{m_co2.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_co2.params['const']      - m_base.params['const'],      m_co2.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_co2.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_co2.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_co2.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_co2.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_co2.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_co2.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_co2.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_co2.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_co2.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_co2.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_co2.params['WML_EUR']    - m_base.params['WML_EUR'],    m_co2.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_co2.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + CO2 joinen
###########################################################################

factors_ts_co2 = factors_ts.join(co2_factor)

# Quintile
reg_data_q5_co2 = (
    quantile_portfolios['Quintile_VW']
    .join(factors_ts_co2)
    .dropna()
)

table_vw_co2_quintile = build_co2_regression_table(
    port_returns     = quantile_portfolios['Quintile_VW'],
    reg_data         = reg_data_q5_co2,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_vw_co2_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_co2 = (
    quantile_portfolios['Terzile_VW']
    .join(factors_ts_co2)
    .dropna()
)

table_vw_co2_terzile = build_co2_regression_table(
    port_returns     = quantile_portfolios['Terzile_VW'],
    reg_data         = reg_data_t3_co2,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_vw_co2_terzile.index.name = 'Terzile'

# Branchen
reg_data_ind_co2 = (
    industry_returns
    .join(factors_ts_co2)
    .dropna()
)

table_vw_co2_industry = build_co2_regression_table(
    port_returns     = industry_returns,
    reg_data         = reg_data_ind_co2,
    factors          = factors,
    median_intensity = median_industry,
)
table_vw_co2_industry.index.name = 'Industry_VW'
table_vw_co2_industry = table_co2_industry.sort_values('CO2', ascending=False)

In [128]:
###########################################################################
# Regressionstabelle – Makrofaktor - Emission Weighted
###########################################################################

def build_co2_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags              = int(len(reg_data) ** (1 / 3))
    factors_with_co2  = factors + ['CO2_Ret']
    rows              = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_co2  = sm.add_constant(reg_data[factors_with_co2].astype(float))
        m_co2  = sm.OLS(y, X_co2).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_co2, y, X_base, X_co2)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_co2.params['const'],      m_co2.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_co2.params['Mkt-RF_EUR'], m_co2.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_co2.params['5 SMB_EUR'],  m_co2.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_co2.params['5 HML_EUR'],  m_co2.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_co2.params['5 RMW_EUR'],  m_co2.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_co2.params['5 CMA_EUR'],  m_co2.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_co2.params['WML_EUR'],    m_co2.pvalues['WML_EUR']),
            'CO2':               fmt_coef(m_co2.params.get('CO2_Ret', np.nan), m_co2.pvalues.get('CO2_Ret', 1)),
            'Adj. R² (%)':       f'{m_co2.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_co2.params['const']      - m_base.params['const'],      m_co2.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_co2.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_co2.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_co2.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_co2.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_co2.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_co2.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_co2.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_co2.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_co2.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_co2.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_co2.params['WML_EUR']    - m_base.params['WML_EUR'],    m_co2.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_co2.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + CO2 joinen
###########################################################################

factors_ts_co2 = factors_ts.join(co2_factor)

# Quintile
reg_data_q5_co2 = (
    quantile_portfolios['Quintile_EW_Emission']
    .join(factors_ts_co2)
    .dropna()
)

table_ewe_co2_quintile = build_co2_regression_table(
    port_returns     = quantile_portfolios['Quintile_EW_Emission'],
    reg_data         = reg_data_q5_co2,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_ewe_co2_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_co2 = (
    quantile_portfolios['Terzile_EW_Emission']
    .join(factors_ts_co2)
    .dropna()
)

table_ewe_co2_terzile = build_co2_regression_table(
    port_returns     = quantile_portfolios['Terzile_EW_Emission'],
    reg_data         = reg_data_t3_co2,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_ewe_co2_terzile.index.name = 'Terzile'

# Branchen
reg_data_ind_co2 = (
    industry_returns
    .join(factors_ts_co2)
    .dropna()
)

table_ewe_co2_industry = build_co2_regression_table(
    port_returns     = industry_returns,
    reg_data         = reg_data_ind_co2,
    factors          = factors,
    median_intensity = median_industry,
)
table_ewe_co2_industry.index.name = 'Industry_EW_Emission'
table_ewe_co2_industry = table_co2_industry.sort_values('CO2', ascending=False)

In [129]:
###########################################################################
# Regressionstabelle – FF5+MOM vs FF5+MOM+BMG+CO2
###########################################################################

def build_bmg_co2_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags                  = int(len(reg_data) ** (1 / 3))
    factors_with_bmg_co2  = factors + ['BMG', 'CO2_Ret']
    rows                  = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_bmg_co2].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_full.params['const'],      m_full.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_full.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_full.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_full.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_full.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_full.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_full.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            'BMG':               fmt_coef(m_full.params.get('BMG',     np.nan), m_full.pvalues.get('BMG',     1)),
            'CO2':               fmt_coef(m_full.params.get('CO2_Ret', np.nan), m_full.pvalues.get('CO2_Ret', 1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_full.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_full.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}', 
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + BMG + CO2 joinen
###########################################################################

factors_ts_full = factors_ts.join(co2_factor).join(bmg_series.rename('BMG'))

# Quintile
reg_data_q5_full = (
    quantile_portfolios['Quintile_EW']
    .join(factors_ts_full)
    .dropna()
)

table_full_quintile = build_bmg_co2_regression_table(
    port_returns     = quantile_portfolios['Quintile_EW'],
    reg_data         = reg_data_q5_full,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_full_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_full = (
    quantile_portfolios['Terzile_EW']
    .join(factors_ts_full)
    .dropna()
)

table_full_terzile = build_bmg_co2_regression_table(
    port_returns     = quantile_portfolios['Terzile_EW'],
    reg_data         = reg_data_t3_full,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_full_terzile.index.name = 'Terzile'

# Branchen
reg_data_ind_full = (
    industry_returns
    .join(factors_ts_full)
    .dropna()
)

table_full_industry = build_bmg_co2_regression_table(
    port_returns     = industry_returns,
    reg_data         = reg_data_ind_full,
    factors          = factors,
    median_intensity = median_industry,
)
table_full_industry.index.name = 'Industry'
table_full_industry = table_full_industry.sort_values('BMG', ascending=False)

In [130]:
###########################################################################
# Regressionstabelle – FF5+MOM vs FF5+MOM+BMG+CO2 - value weighted
###########################################################################

def build_bmg_co2_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags                  = int(len(reg_data) ** (1 / 3))
    factors_with_bmg_co2  = factors + ['BMG', 'CO2_Ret']
    rows                  = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_bmg_co2].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_full.params['const'],      m_full.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_full.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_full.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_full.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_full.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_full.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_full.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            'BMG':               fmt_coef(m_full.params.get('BMG',     np.nan), m_full.pvalues.get('BMG',     1)),
            'CO2':               fmt_coef(m_full.params.get('CO2_Ret', np.nan), m_full.pvalues.get('CO2_Ret', 1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_full.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_full.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}', 
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + BMG + CO2 joinen
###########################################################################

factors_ts_full = factors_ts.join(co2_factor).join(bmg_series.rename('BMG'))

# Quintile
reg_data_q5_full = (
    quantile_portfolios['Quintile_VW']
    .join(factors_ts_full)
    .dropna()
)

table_vw_full_quintile = build_bmg_co2_regression_table(
    port_returns     = quantile_portfolios['Quintile_VW'],
    reg_data         = reg_data_q5_full,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_vw_full_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_full = (
    quantile_portfolios['Terzile_VW']
    .join(factors_ts_full)
    .dropna()
)

table_vw_full_terzile = build_bmg_co2_regression_table(
    port_returns     = quantile_portfolios['Terzile_VW'],
    reg_data         = reg_data_t3_full,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_vw_full_terzile.index.name = 'Terzile'

# Branchen
reg_data_ind_full = (
    industry_returns
    .join(factors_ts_full)
    .dropna()
)

table_vw_full_industry = build_bmg_co2_regression_table(
    port_returns     = industry_returns,
    reg_data         = reg_data_ind_full,
    factors          = factors,
    median_intensity = median_industry,
)
table_vw_full_industry.index.name = 'Industry_VW'
table_vw_full_industry = table_vw_full_industry.sort_values('BMG', ascending=False)

In [131]:
###########################################################################
# Regressionstabelle – Sub-Periode 2021+ mit BMG_Intensity + BMG_Changes + CO2
###########################################################################

def build_triple_factor_subperiod_table(port_returns, reg_data, factors,
                                         start_date, median_intensity=None):

    def sig_stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.10: return '*'
        return ''

    def fmt_coef(coef, p):
        return f'{coef:.4f}{sig_stars(p)}'

    reg_data = reg_data[reg_data.index >= start_date].copy()

    lags             = int(len(reg_data) ** (1 / 3))
    factors_with_all = factors + ['BMG', 'BMG_Changes', 'CO2_Ret']
    rows             = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_all].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':             fmt_coef(m_full.params['const'],      m_full.pvalues['const']),
            'Mkt-RF':            fmt_coef(m_full.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            'SMB':               fmt_coef(m_full.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            'HML':               fmt_coef(m_full.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            'RMW':               fmt_coef(m_full.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            'CMA':               fmt_coef(m_full.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            'WML':               fmt_coef(m_full.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            'BMG (Intensity)':   fmt_coef(m_full.params.get('BMG',         np.nan), m_full.pvalues.get('BMG',         1)),
            'BMG (Changes)':     fmt_coef(m_full.params.get('BMG_Changes', np.nan), m_full.pvalues.get('BMG_Changes', 1)),
            'CO2':               fmt_coef(m_full.params.get('CO2_Ret',     np.nan), m_full.pvalues.get('CO2_Ret',     1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':           fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':          fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':             fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            '∆ HML':             fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            '∆ RMW':             fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':             fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
            'N':                 len(reg_data),
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + alle drei Faktoren joinen
###########################################################################

bmg_changes_series = all_bmg_factors['Scope12_Changes'].rename('BMG_Changes')

factors_ts_triple = (
    factors_ts
    .join(co2_factor)
    .join(all_bmg_factors['Scope_Total_Intensity'].rename('BMG'))
    .join(bmg_changes_series)
)

START_DATE = '2021-01-01'

# Quintile VW
reg_data_q5 = quantile_portfolios['Quintile_VW'].join(factors_ts_triple).dropna()

table_sub_quintile = build_triple_factor_subperiod_table(
    port_returns     = quantile_portfolios['Quintile_VW'],
    reg_data         = reg_data_q5,
    factors          = factors,
    start_date       = START_DATE,
    median_intensity = median_q5.to_dict(),
)
table_sub_quintile.index.name = 'Quintile'

# Terzile VW
reg_data_t3 = quantile_portfolios['Terzile_VW'].join(factors_ts_triple).dropna()

table_sub_terzile = build_triple_factor_subperiod_table(
    port_returns     = quantile_portfolios['Terzile_VW'],
    reg_data         = reg_data_t3,
    factors          = factors,
    start_date       = START_DATE,
    median_intensity = median_t3.to_dict(),
)
table_sub_terzile.index.name = 'Terzile'

# Branchen VW
industry_returns_vw = build_industry_portfolios(df, industry_col='Industry', weighting='value')
reg_data_ind = industry_returns_vw.join(factors_ts_triple).dropna()

table_sub_industry = build_triple_factor_subperiod_table(
    port_returns     = industry_returns_vw,
    reg_data         = reg_data_ind,
    factors          = factors,
    start_date       = START_DATE,
    median_intensity = median_industry,
)
table_sub_industry.index.name = 'Industry'
table_sub_industry = table_sub_industry.sort_values('BMG (Intensity)', ascending=False)

In [132]:
###########################################################################
# Regressionstabelle – FF5+MOM vs FF5+MOM+BMG+CO2 - emission weighted
###########################################################################

def build_bmg_co2_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags                  = int(len(reg_data) ** (1 / 3))
    factors_with_bmg_co2  = factors + ['BMG', 'CO2_Ret']
    rows                  = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_bmg_co2].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_full.params['const'],      m_full.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_full.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_full.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_full.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_full.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_full.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_full.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            'BMG':               fmt_coef(m_full.params.get('BMG',     np.nan), m_full.pvalues.get('BMG',     1)),
            'CO2':               fmt_coef(m_full.params.get('CO2_Ret', np.nan), m_full.pvalues.get('CO2_Ret', 1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_full.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_full.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}', 
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + BMG + CO2 joinen
###########################################################################

factors_ts_full = factors_ts.join(co2_factor).join(bmg_series.rename('BMG'))

# Quintile
reg_data_q5_full = (
    quantile_portfolios['Quintile_EW_Emission']
    .join(factors_ts_full)
    .dropna()
)

table_ewe_full_quintile = build_bmg_co2_regression_table(
    port_returns     = quantile_portfolios['Quintile_EW_Emission'],
    reg_data         = reg_data_q5_full,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_ewe_full_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_full = (
    quantile_portfolios['Terzile_VW']
    .join(factors_ts_full)
    .dropna()
)

table_ewe_full_terzile = build_bmg_co2_regression_table(
    port_returns     = quantile_portfolios['Terzile_EW_Emission'],
    reg_data         = reg_data_t3_full,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_ewe_full_terzile.index.name = 'Terzile'

# Branchen
reg_data_ind_full = (
    industry_returns
    .join(factors_ts_full)
    .dropna()
)

table_ewe_full_industry = build_bmg_co2_regression_table(
    port_returns     = industry_returns,
    reg_data         = reg_data_ind_full,
    factors          = factors,
    median_intensity = median_industry,
)
table_ewe_full_industry.index.name = 'Industry_EW_Emission'
table_ewe_full_industry = table_ewe_full_industry.sort_values('BMG', ascending=False)

In [133]:
###########################################################################
# CO2 Excess Return ~ FF5 + MOM (Gesamt & Sub-Perioden)
###########################################################################

def _stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def _fmt(coef, p):
    return f'{coef:.4f}{_stars(p)}'

SPLIT_DATE = '2019-01'

ts = (
    df[['Date', 'EEX-EU CO2 Emissions'] + factors + ['RF_EUR']]
    .drop_duplicates('Date')
    .dropna()
    .assign(Date=lambda x: to_datetime_safe(x['Date']))
    .sort_values('Date')
)

y_full  = ts.set_index('Date')['EEX-EU CO2 Emissions'] - ts.set_index('Date')['RF_EUR']
y_pre   = y_full[y_full.index <  SPLIT_DATE]
y_post  = y_full[y_full.index >= SPLIT_DATE]

def get_X(subset):
    return sm.add_constant(subset[factors].astype(float))

def fit_model(y, ts_subset):
    lags = int(len(y) ** (1 / 3))
    X    = get_X(ts_subset.set_index('Date'))
    return sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

m_full = fit_model(y_full, ts)
m_pre  = fit_model(y_pre,  ts[ts['Date'] <  SPLIT_DATE])
m_post = fit_model(y_post, ts[ts['Date'] >= SPLIT_DATE])

factor_rename = {
    'const':      'Constant',
    'Mkt-RF_EUR': 'Mkt-RF_EUR',
    '5 SMB_EUR':  '5 SMB_EUR',
    '5 HML_EUR':  '5 HML_EUR',
    '5 RMW_EUR':  '5 RMW_EUR',
    '5 CMA_EUR':  '5 CMA_EUR',
    'WML_EUR':    'WML_EUR',
}

models = {
    f'(1) Gesamt (T={len(y_full)})':    m_full,
    f'(2) Pre-2019 (T={len(y_pre)})':   m_pre,
    f'(3) Post-2019 (T={len(y_post)})': m_post,
}

# Koeffizient + t-Stat in Klammern – alternierend pro Faktor
rows = {}
for param, label in factor_rename.items():
    # Koeffizient-Zeile
    rows[label] = {
        col: _fmt(m.params.get(param, np.nan), m.pvalues.get(param, 1))
        for col, m in models.items()
    }
    # t-Stat-Zeile in Klammern
    rows[f'({label})'] = {
        col: f'({m.tvalues.get(param, np.nan):.3f})'
        for col, m in models.items()
    }

rows['N']       = {col: len(m.fittedvalues)                      for col, m in models.items()}
rows['R²']      = {col: round(m.rsquared, 3)                     for col, m in models.items()}
rows['Adj. R²'] = {col: round(m.rsquared_adj, 3)                 for col, m in models.items()}
rows['F-stat']  = {col: _fmt(m.fvalue, m.f_pvalue)      for col, m in models.items()}

df_co2_table = pd.DataFrame(rows).T


In [134]:
###########################################################################
# MODUL 1 – Zeitreihen Export: Standard vs. Industry-Neutral BMG
###########################################################################
def _sig(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

def build_bmg_comparison_ts(bmg_dict_std, bmg_dict_in):
    """Erstellt Zeitreihen-DataFrame für Standard und Industry-Neutral BMG."""

    def norm100(s):
        cum = 100 * (1 + s.dropna()).cumprod()
        start = pd.Series(
            [100],
            index=[cum.index[0] - pd.DateOffset(months=1)],
            name=cum.name
        )
        return pd.concat([start, cum])

    # Kumulierte Returns für alle Metriken
    std_cum = pd.DataFrame({
        f'{metric} (Std)': norm100(series)
        for metric, series in bmg_dict_std.items()
    })

    in_cum = pd.DataFrame({
        f'{metric} (IN)': norm100(series)
        for metric, series in bmg_dict_in.items()
    })

    # Rohe Returns für alle Metriken
    std_raw = pd.DataFrame({
        f'{metric} (Std)': series
        for metric, series in bmg_dict_std.items()
    })

    in_raw = pd.DataFrame({
        f'{metric} (IN)': series
        for metric, series in bmg_dict_in.items()
    })

    return std_cum.join(in_cum, how='outer'), std_raw.join(in_raw, how='outer')


bmg_cum_ts, bmg_raw_ts = build_bmg_comparison_ts(
    bmg_dict_std = all_bmg_factors,
    bmg_dict_in  = all_bmg_industry_neutral,
)

# --- Übersichtstabelle: Faktor-Charakteristika ---
def build_factor_characteristics(bmg_dict_std, bmg_dict_in, df_factors, factors):

    factors_ts = (
        df_factors[['Date'] + factors].drop_duplicates('Date').copy()
        .assign(Date=lambda x: to_datetime_safe(x['Date']))
        .set_index('Date')
    )

    rows = {}

    for label, series in {
        **{f'BMG Std ({k})': v for k, v in bmg_dict_std.items()},
        **{f'BMG IN ({k})':  v for k, v in bmg_dict_in.items()},
    }.items():

        ret   = series.dropna()
        n     = len(ret)
        mean  = ret.mean()
        std   = ret.std()
        lags  = int(n ** (1/3))

        # HAC t-Stat
        X     = sm.add_constant(pd.Series(np.ones(n), index=ret.index))
        m     = sm.OLS(ret.values, X.values).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
        t_hac = m.tvalues[0]
        p_hac = m.pvalues[0]

        rf_ts = (
            df_factors[['Date', 'RF_EUR']].drop_duplicates('Date').copy()
            .assign(Date=lambda x: to_datetime_safe(x['Date']))
            .set_index('Date')['RF_EUR']
        )
        rf    = rf_ts.reindex(ret.index).fillna(0)
        sharpe = (ret - rf).mean() / std * np.sqrt(12)

        rows[label] = {
            'Mean (% p.m.)':    round(mean * 100, 3),
            'Sig.':             _sig(p_hac),
            't-Stat (HAC)':     round(t_hac, 3),
            'Std. Dev. (%)':    round(std * 100, 3),
            'Sharpe (ann.)':    round(sharpe, 3),
            'N':                n,
        }

    return pd.DataFrame(rows).T

char_table = build_factor_characteristics(
    bmg_dict_std = all_bmg_factors,
    bmg_dict_in  = all_bmg_industry_neutral,
    df_factors   = df,
    factors      = factors,
)

C:\Users\kuhfu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\tsa\tsatools.py:162: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  x = pd.concat(x[::order], axis=1)
C:\Users\kuhfu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\tsa\tsatools.py:162: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  x = pd.concat(x[::order], axis=1)
C:\Users\kuhfu\AppData\Local\Packages\PythonSoftwareFoun

In [135]:
###########################################################################
# MODUL 2 – Spanning Regression Table (Meyer Table 18)
###########################################################################
# Struktur: Zeilen = BMG-Varianten, Spalten = Alpha + Faktor-Betas + Adj.R²
# t-Stats in Klammern unter jedem Koeffizienten

def _sig(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

def _fc(coef, p, dec=4):
    return f'{coef:.{dec}f}{_sig(p)}'

def _tc(t):
    return f'({t:.3f})'

def build_spanning_table(bmg_dict, df_factors, factors, label_suffix=''):

    factors_ts = (
        df_factors[['Date'] + factors].drop_duplicates('Date').copy()
        .assign(Date=lambda x: to_datetime_safe(x['Date']))
        .set_index('Date')
    )

    col_order = ['Alpha'] + factors + ['Adj. R²', 'N']
    rows      = {}

    for metric, bmg_series in bmg_dict.items():
        reg_data = (
            bmg_series.rename('BMG')
            .to_frame()
            .join(factors_ts)
            .dropna()
        )

        y    = reg_data['BMG'].astype(float)
        X    = sm.add_constant(reg_data[factors].astype(float))
        lags = int(len(reg_data) ** (1/3))
        m    = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        label = f'{metric}{label_suffix}'

        # Koeffizienten-Zeile
        rows[label] = {
            'Alpha':   _fc(m.params['const'], m.pvalues['const']),
            **{fac:    _fc(m.params[fac], m.pvalues[fac]) for fac in factors},
            'Adj. R²': f'{m.rsquared_adj * 100:.2f}%',
            'N':       len(reg_data),
        }

        # t-Stat-Zeile
        rows[f'  '] = {   # Leerzeile mit t-Stats
            'Alpha': _tc(m.tvalues['const']),
            **{fac:  _tc(m.tvalues[fac]) for fac in factors},
            'Adj. R²': '',
            'N':       '',
        }

    # Saubere Zeilenreihenfolge
    result = pd.DataFrame(rows).T[col_order]
    return result


# Spanning: Standard BMG
spanning_std = build_spanning_table(all_bmg_factors,          df, factors, label_suffix=' (Std)')
spanning_in  = build_spanning_table(all_bmg_industry_neutral, df, factors, label_suffix=' (IN)')
spanning_all = pd.concat([spanning_std, spanning_in])


In [136]:
###########################################################################
# Charakteristika-Tabelle: EW vs. VW vs. EW-Emission
###########################################################################

def normalize_to_100(cum_returns):
    """Setzt den ersten Wert auf 100 – unterstützt Series und DataFrame."""
    if isinstance(cum_returns, pd.Series):
        start = pd.Series(
            [100],
            index=[cum_returns.index[0] - pd.DateOffset(months=1)],
            name=cum_returns.name
        )
    else:
        start = pd.DataFrame(
            100,
            index   = [cum_returns.index[0] - pd.DateOffset(months=1)],
            columns = cum_returns.columns
        )
    return pd.concat([start, cum_returns])

def build_weighting_comparison(quantile_portfolios, configs_compare, rf_ts):
    """
    Vergleicht Mean, Std, t-Stat und Sharpe für verschiedene Gewichtungen.
    Fokus auf LS-Portfolio als Zusammenfassung pro Variante.
    """
    rows = {}

    for label in configs_compare:
        port = quantile_portfolios[label]

        for col in port.columns:
            ret     = port[col].dropna()
            rf      = rf_ts.reindex(ret.index).fillna(0)
            n       = len(ret)
            mean    = ret.mean()
            std     = ret.std()
            t_stat  = mean / (std / np.sqrt(n))
            p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n - 1))
            sharpe  = (ret - rf).mean() / std * np.sqrt(12)

            rows[f'{label} | {col}'] = {
                'Weighting':        label,
                'Portfolio':        col,
                'Mean (% p.m.)':    round(mean * 100, 4),
                'Std. Dev. (%)':    round(std  * 100, 4),
                't-Statistic':      round(t_stat, 3),
                'p-Value':          round(p_value, 3),
                'Sig.':             '***' if p_value < 0.01 else
                                    '**'  if p_value < 0.05 else
                                    '*'   if p_value < 0.10 else '',
                'Sharpe (ann.)':    round(sharpe, 3),
                'N':                n,
            }

    return pd.DataFrame(rows).T


configs_compare = [
    'Quintile_EW',
    'Quintile_VW',
    'Quintile_EW_Emission',
    'Terzile_EW',
    'Terzile_VW',
    'Terzile_EW_Emission',
]

weighting_comparison = build_weighting_comparison(
    quantile_portfolios = quantile_portfolios,
    configs_compare     = configs_compare,
    rf_ts               = rf_ts,
)

# Kumulierte Returns für Excel
cum_comparison = pd.DataFrame({
    label: normalize_to_100(100 * (1 + quantile_portfolios[label]['LS']).cumprod())
    for label in configs_compare
})

In [137]:
###########################################################################
# Korrelation der BMG-Faktoren untereinander
###########################################################################

# Labels kürzen für bessere Lesbarkeit
bmg_labels = {
    'Scope_Total_Intensity': 'Total Intensity',
    'Scope12_Intensity':     'Scope12 Intensity',
    'Scope_Total':           'Total Absolute',
    'Scope12':               'Scope12 Absolute',
    'ScopeTotal_Changes':    'Total Changes',
    'Scope12_Changes':       'Scope12 Changes',
}

bmg_df = pd.DataFrame({
    bmg_labels[k]: v for k, v in all_bmg_factors.items()
}).dropna()

# Korrelationsmatrix
bmg_all_corr = bmg_df.corr().round(3)

In [138]:
###########################################################################
# Regressionstabelle – FF5+MOM vs FF5+MOM+BMG_Intensity+BMG_Changes
###########################################################################

def sig_stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.4f}{sig_stars(p)}'

def build_dual_bmg_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags                   = int(len(reg_data) ** (1 / 3))
    factors_with_dual_bmg  = factors + ['BMG_Intensity', 'BMG_Changes']
    rows                   = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_dual_bmg].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':             fmt_coef(m_full.params['const'],              m_full.pvalues['const']),
            'Mkt-RF':            fmt_coef(m_full.params['Mkt-RF_EUR'],         m_full.pvalues['Mkt-RF_EUR']),
            'SMB':               fmt_coef(m_full.params['5 SMB_EUR'],          m_full.pvalues['5 SMB_EUR']),
            'HML':               fmt_coef(m_full.params['5 HML_EUR'],          m_full.pvalues['5 HML_EUR']),
            'RMW':               fmt_coef(m_full.params['5 RMW_EUR'],          m_full.pvalues['5 RMW_EUR']),
            'CMA':               fmt_coef(m_full.params['5 CMA_EUR'],          m_full.pvalues['5 CMA_EUR']),
            'WML':               fmt_coef(m_full.params['WML_EUR'],            m_full.pvalues['WML_EUR']),
            'BMG (Intensity)':   fmt_coef(m_full.params.get('BMG_Intensity', np.nan), m_full.pvalues.get('BMG_Intensity', 1)),
            'BMG (Changes)':     fmt_coef(m_full.params.get('BMG_Changes',   np.nan), m_full.pvalues.get('BMG_Changes',   1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':           fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':          fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':             fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            '∆ HML':             fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            '∆ RMW':             fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':             fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + BMG_Intensity + BMG_Changes joinen
###########################################################################

bmg_intensity = all_bmg_factors['Scope12_Intensity'].rename('BMG_Intensity')
bmg_changes   = all_bmg_factors['Scope12_Changes'].rename('BMG_Changes')

factors_ts_dual = factors_ts.join(bmg_intensity).join(bmg_changes)

# Quintile
reg_data_q5_dual = (
    quantile_portfolios['Quintile_VW']
    .join(factors_ts_dual)
    .dropna()
)

table_dual_quintile = build_dual_bmg_regression_table(
    port_returns     = quantile_portfolios['Quintile_VW'],
    reg_data         = reg_data_q5_dual,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_dual_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_dual = (
    quantile_portfolios['Terzile_VW']
    .join(factors_ts_dual)
    .dropna()
)

table_dual_terzile = build_dual_bmg_regression_table(
    port_returns     = quantile_portfolios['Terzile_VW'],
    reg_data         = reg_data_t3_dual,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_dual_terzile.index.name = 'Terzile'

# Branchen
reg_data_ind_dual = (
    industry_returns
    .join(factors_ts_dual)
    .dropna()
)

table_dual_industry = build_dual_bmg_regression_table(
    port_returns     = industry_returns,
    reg_data         = reg_data_ind_dual,
    factors          = factors,
    median_intensity = median_industry,
)
table_dual_industry.index.name  = 'Industry'
table_dual_industry = table_dual_industry.sort_values('BMG (Intensity)', ascending=False)

In [139]:
###########################################################################
# Regressionstabelle – FF5+MOM vs FF5+MOM+BMG_Intensity+BMG_Absolute+BMG_Changes
###########################################################################

def build_triple_bmg_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags                    = int(len(reg_data) ** (1 / 3))
    factors_with_triple_bmg = factors + ['BMG_Intensity', 'BMG_Absolute', 'BMG_Changes']
    rows                    = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_triple_bmg].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':             fmt_coef(m_full.params['const'],              m_full.pvalues['const']),
            'Mkt-RF':            fmt_coef(m_full.params['Mkt-RF_EUR'],         m_full.pvalues['Mkt-RF_EUR']),
            'SMB':               fmt_coef(m_full.params['5 SMB_EUR'],          m_full.pvalues['5 SMB_EUR']),
            'HML':               fmt_coef(m_full.params['5 HML_EUR'],          m_full.pvalues['5 HML_EUR']),
            'RMW':               fmt_coef(m_full.params['5 RMW_EUR'],          m_full.pvalues['5 RMW_EUR']),
            'CMA':               fmt_coef(m_full.params['5 CMA_EUR'],          m_full.pvalues['5 CMA_EUR']),
            'WML':               fmt_coef(m_full.params['WML_EUR'],            m_full.pvalues['WML_EUR']),
            'BMG (Intensity)':   fmt_coef(m_full.params.get('BMG_Intensity', np.nan), m_full.pvalues.get('BMG_Intensity', 1)),
            'BMG (Absolute)':    fmt_coef(m_full.params.get('BMG_Absolute',  np.nan), m_full.pvalues.get('BMG_Absolute',  1)),
            'BMG (Changes)':     fmt_coef(m_full.params.get('BMG_Changes',   np.nan), m_full.pvalues.get('BMG_Changes',   1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':           fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':          fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':             fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            '∆ HML':             fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            '∆ RMW':             fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':             fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + alle drei BMG-Faktoren joinen
###########################################################################

bmg_intensity = all_bmg_factors['Scope12_Intensity'].rename('BMG_Intensity')
bmg_absolute  = all_bmg_factors['Scope12'].rename('BMG_Absolute')
bmg_changes   = all_bmg_factors['Scope12_Changes'].rename('BMG_Changes')

factors_ts_triple = factors_ts.join(bmg_intensity).join(bmg_absolute).join(bmg_changes)

# Quintile
reg_data_q5_triple = (
    quantile_portfolios['Quintile_EW']
    .join(factors_ts_triple)
    .dropna()
)

table_triple_quintile = build_triple_bmg_regression_table(
    port_returns     = quantile_portfolios['Quintile_EW'],
    reg_data         = reg_data_q5_triple,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_triple_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_triple = (
    quantile_portfolios['Terzile_EW']
    .join(factors_ts_triple)
    .dropna()
)

table_triple_terzile = build_triple_bmg_regression_table(
    port_returns     = quantile_portfolios['Terzile_EW'],
    reg_data         = reg_data_t3_triple,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_triple_terzile.index.name = 'Terzile'

# Branchen
reg_data_ind_triple = (
    industry_returns
    .join(factors_ts_triple)
    .dropna()
)

table_triple_industry = build_triple_bmg_regression_table(
    port_returns     = industry_returns,
    reg_data         = reg_data_ind_triple,
    factors          = factors,
    median_intensity = median_industry,
)
table_triple_industry.index.name = 'Industry'
table_triple_industry = table_triple_industry.sort_values('BMG (Intensity)', ascending=False)

In [140]:
###########################################################################
# Regressionstabelle – FF5+MOM vs FF5+MOM+BMG+CO2+BMG_Changes - value-weighted
###########################################################################

def sig_stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.4f}{sig_stars(p)}'

def build_bmg_co2_changes_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags                   = int(len(reg_data) ** (1 / 3))
    factors_with_all       = factors + ['BMG', 'CO2_Ret', 'BMG_Changes']
    rows                   = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_all].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':             fmt_coef(m_full.params['const'],      m_full.pvalues['const']),
            'Mkt-RF':            fmt_coef(m_full.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            'SMB':               fmt_coef(m_full.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            'HML':               fmt_coef(m_full.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            'RMW':               fmt_coef(m_full.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            'CMA':               fmt_coef(m_full.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            'WML':               fmt_coef(m_full.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            'BMG (Intensity)':   fmt_coef(m_full.params.get('BMG',         np.nan), m_full.pvalues.get('BMG',         1)),
            'CO2':               fmt_coef(m_full.params.get('CO2_Ret',     np.nan), m_full.pvalues.get('CO2_Ret',     1)),
            'BMG (Changes)':     fmt_coef(m_full.params.get('BMG_Changes', np.nan), m_full.pvalues.get('BMG_Changes', 1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':           fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':          fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':             fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            '∆ HML':             fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            '∆ RMW':             fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':             fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + BMG + CO2 + BMG_Changes joinen
###########################################################################

bmg_changes_series = all_bmg_factors['Scope12_Changes'].rename('BMG_Changes')

factors_ts_full_changes = (
    factors_ts
    .join(co2_factor)
    .join(bmg_series.rename('BMG'))
    .join(bmg_changes_series)
)

# Quintile
reg_data_q5_full_changes = (
    quantile_portfolios['Quintile_VW']
    .join(factors_ts_full_changes)
    .dropna()
)

table_vw_full_changes_quintile = build_bmg_co2_changes_regression_table(
    port_returns     = quantile_portfolios['Quintile_VW'],
    reg_data         = reg_data_q5_full_changes,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_vw_full_changes_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_full_changes = (
    quantile_portfolios['Terzile_VW']
    .join(factors_ts_full_changes)
    .dropna()
)

table_vw_full_changes_terzile = build_bmg_co2_changes_regression_table(
    port_returns     = quantile_portfolios['Terzile_VW'],
    reg_data         = reg_data_t3_full_changes,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_vw_full_changes_terzile.index.name = 'Terzile'

industry_returns_vw = build_industry_portfolios(df, industry_col='Industry', weighting='value')

# Branchen
reg_data_ind_full_changes = (
    industry_returns_vw
    .join(factors_ts_full_changes)
    .dropna()
)

table_vw_full_changes_industry = build_bmg_co2_changes_regression_table(
    port_returns     = industry_returns_vw,
    reg_data         = reg_data_ind_full_changes,
    factors          = factors,
    median_intensity = median_industry,
)
table_vw_full_changes_industry.index.name = 'Industry'
table_vw_full_changes_industry = table_vw_full_changes_industry.sort_values('BMG (Intensity)', ascending=False)

In [141]:
###########################################################################
# Regressionstabelle – FF5+MOM vs FF5+MOM+BMG+CO2+BMG_Changes - equal-weighted
###########################################################################

def sig_stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.4f}{sig_stars(p)}'

def build_bmg_co2_changes_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags                   = int(len(reg_data) ** (1 / 3))
    factors_with_all       = factors + ['BMG', 'CO2_Ret', 'BMG_Changes']
    rows                   = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_all].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':             fmt_coef(m_full.params['const'],      m_full.pvalues['const']),
            'Mkt-RF':            fmt_coef(m_full.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            'SMB':               fmt_coef(m_full.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            'HML':               fmt_coef(m_full.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            'RMW':               fmt_coef(m_full.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            'CMA':               fmt_coef(m_full.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            'WML':               fmt_coef(m_full.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            'BMG (Intensity)':   fmt_coef(m_full.params.get('BMG',         np.nan), m_full.pvalues.get('BMG',         1)),
            'CO2':               fmt_coef(m_full.params.get('CO2_Ret',     np.nan), m_full.pvalues.get('CO2_Ret',     1)),
            'BMG (Changes)':     fmt_coef(m_full.params.get('BMG_Changes', np.nan), m_full.pvalues.get('BMG_Changes', 1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':           fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':          fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':             fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            '∆ HML':             fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            '∆ RMW':             fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':             fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + BMG + CO2 + BMG_Changes joinen
###########################################################################

bmg_changes_series = all_bmg_factors['Scope12_Changes'].rename('BMG_Changes')

factors_ts_full_changes = (
    factors_ts
    .join(co2_factor)
    .join(bmg_series.rename('BMG'))
    .join(bmg_changes_series)
)

# Quintile
reg_data_q5_full_changes = (
    quantile_portfolios['Quintile_EW']
    .join(factors_ts_full_changes)
    .dropna()
)

table_ew_full_changes_quintile = build_bmg_co2_changes_regression_table(
    port_returns     = quantile_portfolios['Quintile_EW'],
    reg_data         = reg_data_q5_full_changes,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_ew_full_changes_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_full_changes = (
    quantile_portfolios['Terzile_EW']
    .join(factors_ts_full_changes)
    .dropna()
)

table_ew_full_changes_terzile = build_bmg_co2_changes_regression_table(
    port_returns     = quantile_portfolios['Terzile_EW'],
    reg_data         = reg_data_t3_full_changes,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_ew_full_changes_terzile.index.name = 'Terzile'

industry_returns_ew = build_industry_portfolios(df, industry_col='Industry', weighting='equal')

# Branchen
reg_data_ind_full_changes = (
    industry_returns_ew
    .join(factors_ts_full_changes)
    .dropna()
)

table_ew_full_changes_industry = build_bmg_co2_changes_regression_table(
    port_returns     = industry_returns_ew,
    reg_data         = reg_data_ind_full_changes,
    factors          = factors,
    median_intensity = median_industry,
)
table_ew_full_changes_industry.index.name = 'Industry'
table_ew_full_changes_industry = table_ew_full_changes_industry.sort_values('BMG (Intensity)', ascending=False)

In [142]:
###########################################################################
# Regressionstabelle – FF5+MOM vs FF5+MOM+BMG+CO2+BMG_Changes - emission-weighted
###########################################################################

def sig_stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.4f}{sig_stars(p)}'

def build_bmg_co2_changes_regression_table(port_returns, reg_data, factors, median_intensity=None):

    lags                   = int(len(reg_data) ** (1 / 3))
    factors_with_all       = factors + ['BMG', 'CO2_Ret', 'BMG_Changes']
    rows                   = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_full = sm.add_constant(reg_data[factors_with_all].astype(float))
        m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_full, y, X_base, X_full)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':             fmt_coef(m_full.params['const'],      m_full.pvalues['const']),
            'Mkt-RF':            fmt_coef(m_full.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            'SMB':               fmt_coef(m_full.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            'HML':               fmt_coef(m_full.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            'RMW':               fmt_coef(m_full.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            'CMA':               fmt_coef(m_full.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            'WML':               fmt_coef(m_full.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            'BMG (Intensity)':   fmt_coef(m_full.params.get('BMG',         np.nan), m_full.pvalues.get('BMG',         1)),
            'CO2':               fmt_coef(m_full.params.get('CO2_Ret',     np.nan), m_full.pvalues.get('CO2_Ret',     1)),
            'BMG (Changes)':     fmt_coef(m_full.params.get('BMG_Changes', np.nan), m_full.pvalues.get('BMG_Changes', 1)),
            'Adj. R² (%)':       f'{m_full.rsquared_adj * 100:.2f}%',
            '∆ Alpha':           fmt_coef(m_full.params['const']      - m_base.params['const'],      m_full.pvalues['const']),
            '∆ Mkt-RF':          fmt_coef(m_full.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_full.pvalues['Mkt-RF_EUR']),
            '∆ SMB':             fmt_coef(m_full.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'],  m_full.pvalues['5 SMB_EUR']),
            '∆ HML':             fmt_coef(m_full.params['5 HML_EUR']  - m_base.params['5 HML_EUR'],  m_full.pvalues['5 HML_EUR']),
            '∆ RMW':             fmt_coef(m_full.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'],  m_full.pvalues['5 RMW_EUR']),
            '∆ CMA':             fmt_coef(m_full.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'],  m_full.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_full.params['WML_EUR']    - m_base.params['WML_EUR'],    m_full.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_full.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Reg-Daten: Faktoren + BMG + CO2 + BMG_Changes joinen
###########################################################################

bmg_changes_series = all_bmg_factors['Scope12_Changes'].rename('BMG_Changes')

factors_ts_full_changes = (
    factors_ts
    .join(co2_factor)
    .join(bmg_series.rename('BMG'))
    .join(bmg_changes_series)
)

# Quintile
reg_data_q5_full_changes = (
    quantile_portfolios['Quintile_EW_Emission']
    .join(factors_ts_full_changes)
    .dropna()
)

table_ewe_full_changes_quintile = build_bmg_co2_changes_regression_table(
    port_returns     = quantile_portfolios['Quintile_EW_Emission'],
    reg_data         = reg_data_q5_full_changes,
    factors          = factors,
    median_intensity = median_q5.to_dict(),
)
table_ewe_full_changes_quintile.index.name = 'Quintile'

# Terzile
reg_data_t3_full_changes = (
    quantile_portfolios['Terzile_EW_Emission']
    .join(factors_ts_full_changes)
    .dropna()
)

table_ewe_full_changes_terzile = build_bmg_co2_changes_regression_table(
    port_returns     = quantile_portfolios['Terzile_EW_Emission'],
    reg_data         = reg_data_t3_full_changes,
    factors          = factors,
    median_intensity = median_t3.to_dict(),
)
table_ewe_full_changes_terzile.index.name = 'Terzile'

industry_returns_ewe = build_industry_portfolios(df, industry_col='Industry', weighting='ew_emission')

# Branchen
reg_data_ind_full_changes = (
    industry_returns_ewe
    .join(factors_ts_full_changes)
    .dropna()
)

table_ewe_full_changes_industry = build_bmg_co2_changes_regression_table(
    port_returns     = industry_returns_ewe,
    reg_data         = reg_data_ind_full_changes,
    factors          = factors,
    median_intensity = median_industry,
)
table_ewe_full_changes_industry.index.name = 'Industry'
table_ewe_full_changes_industry = table_ewe_full_changes_industry.sort_values('BMG (Intensity)', ascending=False)

In [143]:
###########################################################################
# Regressionstabelle - BMG Changes - value weightes
###########################################################################

def stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.3f}{stars(p)}'

def build_regression_table(port_returns, reg_data, factors, median_intensity=None):
 
    lags             = int(len(reg_data) ** (1 / 3))
    factors_with_bmg = factors + ['BMG']
    rows             = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_bmg  = sm.add_constant(reg_data[factors_with_bmg].astype(float))
        m_bmg  = sm.OLS(y, X_bmg).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_bmg, y, X_base, X_bmg)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_bmg.params['const'],      m_bmg.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_bmg.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_bmg.params['5 SMB_EUR'],  m_bmg.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_bmg.params['5 HML_EUR'],  m_bmg.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_bmg.params['5 RMW_EUR'],  m_bmg.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_bmg.params['5 CMA_EUR'],  m_bmg.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_bmg.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            'BMG':        fmt_coef(m_bmg.params['BMG'],        m_bmg.pvalues['BMG']),
            'Adj. R² (%)':       f'{m_bmg.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_bmg.params['const']      - m_base.params['const'],      m_bmg.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_bmg.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_bmg.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_bmg.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_bmg.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_bmg.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_bmg.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_bmg.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_bmg.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_bmg.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_bmg.params['WML_EUR']    - m_base.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_bmg.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Faktoren-Zeitreihe (gemeinsam für alle Tabellen)
###########################################################################

factors_ts = (
    df[['Date'] + factors].drop_duplicates('Date').copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .set_index('Date')
)

bmg_series = all_bmg_factors['Scope12_Changes']

###########################################################################
# Tabelle 1 – Quintile EW
###########################################################################

reg_data_q5 = (
    quantile_portfolios['Quintile_VW']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Quintil
median_q5 = (
    df_sorted_plot.dropna(subset=['Bucket', 'Scope12_Intensity'])
                  .groupby('Bucket')['Scope12_Intensity']
                  .median().round(3)
)
median_q5.index = [f'P{int(i)}' for i in median_q5.index]
median_q5['LS'] = np.nan

table_chg_vw_quintile = build_regression_table(
    port_returns      = quantile_portfolios['Quintile_VW'],
    reg_data          = reg_data_q5,
    factors           = factors,
    median_intensity  = median_q5.to_dict(),
)
table_chg_vw_quintile.index.name = 'Quintile_VW'

###########################################################################
# Tabelle 2 – Terzile EW
###########################################################################

reg_data_t3 = (
    quantile_portfolios['Terzile_VW']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Terzil
df_sorted_t3 = build_quantile_portfolios(df, emission_var, n=3, weighting='equal')[1] if False else \
    df_sorted_plot.copy()   

annual_t3 = (
    df[['ISIN', 'Date', emission_var]].copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .assign(Year=lambda x: x['Date'].dt.year)
      .drop_duplicates(subset=['ISIN', 'Year'])
      .dropna(subset=[emission_var])
)
annual_t3['Bucket'] = annual_t3.groupby('Year')[emission_var].transform(
    lambda x: safe_quantile_sort(x, 3)
)

median_t3 = (
    annual_t3.dropna(subset=['Bucket'])
             .groupby('Bucket')[emission_var]
             .median().round(3)
)
median_t3.index = [f'P{int(i)}' for i in median_t3.index]
median_t3['LS'] = np.nan

table_chg_vw_terzile = build_regression_table(
    port_returns      = quantile_portfolios['Terzile_VW'],
    reg_data          = reg_data_t3,
    factors           = factors,
    median_intensity  = median_t3.to_dict(),
)
table_chg_vw_terzile.index.name = 'Terzile_vw'

###########################################################################
# Tabelle 3 – Branchen
###########################################################################

industry_returns_vw = build_industry_portfolios(df, industry_col='Industry', weighting='value')

reg_data_ind = (
    industry_returns_vw
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Branche
median_industry = (
    df.dropna(subset=['Industry', 'Scope12_Intensity'])
      .groupby('Industry')['Scope12_Intensity']
      .median().round(3)
      .to_dict()
)

table_chg_vw_industry = build_regression_table(
    port_returns      = industry_returns_vw,
    reg_data          = reg_data_ind,
    factors           = factors,
    median_intensity  = median_industry,
)
table_chg_vw_industry.index.name = 'Industry_vw'
table_chg_vw_industry = table_chg_vw_industry.sort_values('BMG', ascending=False)

In [144]:
###########################################################################
# Regressionstabelle - BMG Changes - equal weightes
###########################################################################

def stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.3f}{stars(p)}'

def build_regression_table(port_returns, reg_data, factors, median_intensity=None):
 
    lags             = int(len(reg_data) ** (1 / 3))
    factors_with_bmg = factors + ['BMG']
    rows             = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_bmg  = sm.add_constant(reg_data[factors_with_bmg].astype(float))
        m_bmg  = sm.OLS(y, X_bmg).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_bmg, y, X_base, X_bmg)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_bmg.params['const'],      m_bmg.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_bmg.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_bmg.params['5 SMB_EUR'],  m_bmg.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_bmg.params['5 HML_EUR'],  m_bmg.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_bmg.params['5 RMW_EUR'],  m_bmg.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_bmg.params['5 CMA_EUR'],  m_bmg.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_bmg.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            'BMG':        fmt_coef(m_bmg.params['BMG'],        m_bmg.pvalues['BMG']),
            'Adj. R² (%)':       f'{m_bmg.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_bmg.params['const']      - m_base.params['const'],      m_bmg.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_bmg.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_bmg.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_bmg.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_bmg.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_bmg.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_bmg.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_bmg.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_bmg.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_bmg.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_bmg.params['WML_EUR']    - m_base.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_bmg.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Faktoren-Zeitreihe (gemeinsam für alle Tabellen)
###########################################################################

factors_ts = (
    df[['Date'] + factors].drop_duplicates('Date').copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .set_index('Date')
)

bmg_series = all_bmg_factors['Scope12_Changes']

###########################################################################
# Tabelle 1 – Quintile EW
###########################################################################

reg_data_q5 = (
    quantile_portfolios['Quintile_EW']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Quintil
median_q5 = (
    df_sorted_plot.dropna(subset=['Bucket', 'Scope12_Intensity'])
                  .groupby('Bucket')['Scope12_Intensity']
                  .median().round(3)
)
median_q5.index = [f'P{int(i)}' for i in median_q5.index]
median_q5['LS'] = np.nan

table_chg_ew_quintile = build_regression_table(
    port_returns      = quantile_portfolios['Quintile_EW'],
    reg_data          = reg_data_q5,
    factors           = factors,
    median_intensity  = median_q5.to_dict(),
)
table_chg_ew_quintile.index.name = 'Quintile_EW'

###########################################################################
# Tabelle 2 – Terzile EW
###########################################################################

reg_data_t3 = (
    quantile_portfolios['Terzile_EW']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Terzil
df_sorted_t3 = build_quantile_portfolios(df, emission_var, n=3, weighting='equal')[1] if False else \
    df_sorted_plot.copy()   

annual_t3 = (
    df[['ISIN', 'Date', emission_var]].copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .assign(Year=lambda x: x['Date'].dt.year)
      .drop_duplicates(subset=['ISIN', 'Year'])
      .dropna(subset=[emission_var])
)
annual_t3['Bucket'] = annual_t3.groupby('Year')[emission_var].transform(
    lambda x: safe_quantile_sort(x, 3)
)

median_t3 = (
    annual_t3.dropna(subset=['Bucket'])
             .groupby('Bucket')[emission_var]
             .median().round(3)
)
median_t3.index = [f'P{int(i)}' for i in median_t3.index]
median_t3['LS'] = np.nan

table_chg_ew_terzile = build_regression_table(
    port_returns      = quantile_portfolios['Terzile_EW'],
    reg_data          = reg_data_t3,
    factors           = factors,
    median_intensity  = median_t3.to_dict(),
)
table_chg_ew_terzile.index.name = 'Terzile_ew'

###########################################################################
# Tabelle 3 – Branchen
###########################################################################

industry_returns_ew = build_industry_portfolios(df, industry_col='Industry', weighting='equal')

reg_data_ind = (
    industry_returns_ew
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Branche
median_industry = (
    df.dropna(subset=['Industry', 'Scope12_Intensity'])
      .groupby('Industry')['Scope12_Intensity']
      .median().round(3)
      .to_dict()
)

table_chg_ew_industry = build_regression_table(
    port_returns      = industry_returns_ew,
    reg_data          = reg_data_ind,
    factors           = factors,
    median_intensity  = median_industry,
)
table_chg_ew_industry.index.name = 'Industry_ew'
table_chg_ew_industry = table_chg_ew_industry.sort_values('BMG', ascending=False)

In [145]:
###########################################################################
# Regressionstabelle - BMG Changes - emission weighted
###########################################################################

def stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''

def fmt_coef(coef, p):
    return f'{coef:.3f}{stars(p)}'

def build_regression_table(port_returns, reg_data, factors, median_intensity=None):
 
    lags             = int(len(reg_data) ** (1 / 3))
    factors_with_bmg = factors + ['BMG']
    rows             = {}

    for port in port_returns.columns:
        if port not in reg_data.columns:
            continue

        y      = reg_data[port].astype(float)
        X_base = sm.add_constant(reg_data[factors].astype(float))
        m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        X_bmg  = sm.add_constant(reg_data[factors_with_bmg].astype(float))
        m_bmg  = sm.OLS(y, X_bmg).fit(cov_type='HAC', cov_kwds={'maxlags': lags})

        # Nested F-Test für ∆ Adj. R²
        f_stat, f_pval = nested_f_test(m_base, m_bmg, y, X_base, X_bmg)

        rows[port] = {
            'Median Intensity':  median_intensity.get(port, np.nan) if median_intensity else np.nan,
            'Alpha':      fmt_coef(m_bmg.params['const'],      m_bmg.pvalues['const']),
            'Mkt-RF':     fmt_coef(m_bmg.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            'SMB':        fmt_coef(m_bmg.params['5 SMB_EUR'],  m_bmg.pvalues['5 SMB_EUR']),
            'HML':        fmt_coef(m_bmg.params['5 HML_EUR'],  m_bmg.pvalues['5 HML_EUR']),
            'RMW':        fmt_coef(m_bmg.params['5 RMW_EUR'],  m_bmg.pvalues['5 RMW_EUR']),
            'CMA':        fmt_coef(m_bmg.params['5 CMA_EUR'],  m_bmg.pvalues['5 CMA_EUR']),
            'WML':        fmt_coef(m_bmg.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            'BMG':        fmt_coef(m_bmg.params['BMG'],        m_bmg.pvalues['BMG']),
            'Adj. R² (%)':       f'{m_bmg.rsquared_adj * 100:.2f}%',
            '∆ Alpha':    fmt_coef(m_bmg.params['const']      - m_base.params['const'],      m_bmg.pvalues['const']),
            '∆ Mkt-RF':   fmt_coef(m_bmg.params['Mkt-RF_EUR'] - m_base.params['Mkt-RF_EUR'], m_bmg.pvalues['Mkt-RF_EUR']),
            '∆ SMB':      fmt_coef(m_bmg.params['5 SMB_EUR']  - m_base.params['5 SMB_EUR'], m_bmg.pvalues['5 SMB_EUR']),
            '∆ HML':      fmt_coef(m_bmg.params['5 HML_EUR']  - m_base.params['5 HML_EUR'], m_bmg.pvalues['5 HML_EUR']),
            '∆ RMW':      fmt_coef(m_bmg.params['5 RMW_EUR']  - m_base.params['5 RMW_EUR'], m_bmg.pvalues['5 RMW_EUR']),
            '∆ CMA':      fmt_coef(m_bmg.params['5 CMA_EUR']  - m_base.params['5 CMA_EUR'], m_bmg.pvalues['5 CMA_EUR']),
            '∆ WML':             fmt_coef(m_bmg.params['WML_EUR']    - m_base.params['WML_EUR'],    m_bmg.pvalues['WML_EUR']),
            '∆ Adj. R²':         f'{(m_bmg.rsquared_adj - m_base.rsquared_adj) * 100:.2f}%{sig_stars(f_pval)}',
        }

    return pd.DataFrame(rows).T


###########################################################################
# Faktoren-Zeitreihe (gemeinsam für alle Tabellen)
###########################################################################

factors_ts = (
    df[['Date'] + factors].drop_duplicates('Date').copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .set_index('Date')
)

bmg_series = all_bmg_factors['Scope12_Changes']

###########################################################################
# Tabelle 1 – Quintile EW
###########################################################################

reg_data_q5 = (
    quantile_portfolios['Quintile_EW_Emission']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Quintil
median_q5 = (
    df_sorted_plot.dropna(subset=['Bucket', 'Scope12_Intensity'])
                  .groupby('Bucket')['Scope12_Intensity']
                  .median().round(3)
)
median_q5.index = [f'P{int(i)}' for i in median_q5.index]
median_q5['LS'] = np.nan

table_chg_ewe_quintile = build_regression_table(
    port_returns      = quantile_portfolios['Quintile_EW_Emission'],
    reg_data          = reg_data_q5,
    factors           = factors,
    median_intensity  = median_q5.to_dict(),
)
table_chg_ewe_quintile.index.name = 'Quintile_EW__Emission'

###########################################################################
# Tabelle 2 – Terzile EW
###########################################################################

reg_data_t3 = (
    quantile_portfolios['Terzile_EW_Emission']
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Terzil
df_sorted_t3 = build_quantile_portfolios(df, emission_var, n=3, weighting='equal')[1] if False else \
    df_sorted_plot.copy()   

annual_t3 = (
    df[['ISIN', 'Date', emission_var]].copy()
      .assign(Date=lambda x: to_datetime_safe(x['Date']))
      .assign(Year=lambda x: x['Date'].dt.year)
      .drop_duplicates(subset=['ISIN', 'Year'])
      .dropna(subset=[emission_var])
)
annual_t3['Bucket'] = annual_t3.groupby('Year')[emission_var].transform(
    lambda x: safe_quantile_sort(x, 3)
)

median_t3 = (
    annual_t3.dropna(subset=['Bucket'])
             .groupby('Bucket')[emission_var]
             .median().round(3)
)
median_t3.index = [f'P{int(i)}' for i in median_t3.index]
median_t3['LS'] = np.nan

table_chg_ewe_terzile = build_regression_table(
    port_returns      = quantile_portfolios['Terzile_EW_Emission'],
    reg_data          = reg_data_t3,
    factors           = factors,
    median_intensity  = median_t3.to_dict(),
)
table_chg_ewe_terzile.index.name = 'Terzile_ew_Emission'

###########################################################################
# Tabelle 3 – Branchen
###########################################################################

industry_returns_ewe = build_industry_portfolios(df, industry_col='Industry', weighting='emission')

reg_data_ind = (
    industry_returns_ewe
    .join(factors_ts)
    .join(bmg_series.rename('BMG'))
    .dropna()
)

# Median Intensität pro Branche
median_industry = (
    df.dropna(subset=['Industry', 'Scope12_Intensity'])
      .groupby('Industry')['Scope12_Intensity']
      .median().round(3)
      .to_dict()
)

table_chg_ewe_industry = build_regression_table(
    port_returns      = industry_returns_ewe,
    reg_data          = reg_data_ind,
    factors           = factors,
    median_intensity  = median_industry,
)
table_chg_ewe_industry.index.name = 'Industry_ew_Emission'
table_chg_ewe_industry = table_chg_ewe_industry.sort_values('BMG', ascending=False)

In [ ]:
###########################################################################
# Robustheitstest: BMG nach Unternehmensgröße (Big vs. Small)
###########################################################################
# Idee: Gesamtstichprobe wird anhand des jahresweisen Median der Market Cap
# in Big und Small aufgeteilt. Auf jeder Teilstichprobe wird dann die
# bestehende compute_bmg Logik (2×3 Sorting) separat angewendet.
# Frage: Ist der BMG-Effekt größen-abhängig?

def compute_bmg_by_size(df, emission_metric, size_group='Big'):
    """
    Filtert df auf Big oder Small Unternehmen (jahresweiser Median)
    und berechnet den BMG-Faktor auf der Teilstichprobe.

    size_group : 'Big' oder 'Small'
    """
    df = df.copy()
    df['Date'] = df['Date'].dt.to_timestamp() if isinstance(df['Date'].dtype, pd.PeriodDtype) else pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year

    # Jahresweiser Median-Split über die gesamte Stichprobe
    annual_mcap = (
        df[['ISIN', 'Year', 'Company Market Cap']]
        .drop_duplicates(subset=['ISIN', 'Year'])
        .dropna(subset=['Company Market Cap'])
        .copy()
    )
    annual_mcap['Size_Global'] = annual_mcap.groupby('Year')['Company Market Cap'].transform(
        lambda x: np.where(x > x.median(), 'Big', 'Small')
    )

    # ISINs der jeweiligen Größengruppe identifizieren
    big_small_isins = (
        annual_mcap[annual_mcap['Size_Global'] == size_group]
        [['ISIN', 'Year']]
        .drop_duplicates()
    )

    # df auf Teilstichprobe filtern (ISIN-Jahr-Kombination)
    df_filtered = df.merge(big_small_isins, on=['ISIN', 'Year'], how='inner')

    # Bestehende compute_bmg Logik auf Teilstichprobe anwenden
    return compute_bmg(df_filtered, emission_metric)


# Anwendung für alle Metriken
all_bmg_big   = {}
all_bmg_small = {}

for metric in robustness_metrics:
    port_big,   _ = compute_bmg_by_size(df, metric, size_group='Big')
    port_small, _ = compute_bmg_by_size(df, metric, size_group='Small')

    all_bmg_big[metric]   = port_big.set_index('Date')['BMG']
    all_bmg_small[metric] = port_small.set_index('Date')['BMG']

df_bmg_big   = pd.DataFrame(all_bmg_big)
df_bmg_small = pd.DataFrame(all_bmg_small)

# Vergleichstabelle: Charakteristika Big vs. Small vs. Gesamt
def bmg_size_characteristics(bmg_dict, label, df_factors, factors):
    rows = {}
    rf_ts = (
        df_factors[['Date', 'RF_EUR']].drop_duplicates('Date')
        .assign(Date=lambda x: to_datetime_safe(x['Date']))
        .set_index('Date')['RF_EUR']
    )
    for metric, series in bmg_dict.items():
        ret   = series.dropna()
        n     = len(ret)
        mean  = ret.mean()
        std   = ret.std()
        t     = mean / (std / np.sqrt(n))
        p     = 2 * (1 - stats.t.cdf(abs(t), df=n - 1))
        rf    = rf_ts.reindex(ret.index).fillna(0)
        sharpe = (ret - rf).mean() / std * np.sqrt(12)

        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

        rows[f'{label} | {metric}'] = {
            'Group':          label,
            'Metric':         metric,
            'Mean (% p.m.)':  round(mean * 100, 3),
            'Sig.':           sig,
            't-Statistic':    round(t, 3),
            'Std. Dev. (%)':  round(std * 100, 3),
            'Sharpe (ann.)':  round(sharpe, 3),
            'N':              n,
        }
    return pd.DataFrame(rows).T

char_big   = bmg_size_characteristics(all_bmg_big,        'Big',   df, factors)
char_small = bmg_size_characteristics(all_bmg_small,       'Small', df, factors)
char_all   = bmg_size_characteristics(all_bmg_factors,     'All',   df, factors)

size_comparison = pd.concat([char_all, char_big, char_small])

  Big: 521 ISINs
  Small: 502 ISINs
  Big: 521 ISINs
  Small: 502 ISINs
  Big: 521 ISINs
  Small: 502 ISINs
  Big: 521 ISINs
  Small: 502 ISINs
  Big: 521 ISINs
  Small: 502 ISINs
  Big: 521 ISINs
  Small: 502 ISINs


In [ ]:
export_path = OUTPUT / 'descriptive_statistics.xlsx'

with pd.ExcelWriter(export_path, engine='openpyxl') as writer:

    desc_table.to_excel(writer, sheet_name='CI_Industry')
    coverage_table.to_excel(writer, sheet_name='Emission_Coverage', index=False)

    cumulative_returns.to_excel(writer, sheet_name='Cumulative_Returns')
    # cumulative_market_adj.to_excel(writer, sheet_name='Market_Adj_Returns')
    cumulative_returns_terzile.to_excel(writer,       sheet_name='Cumulative_Returns_Terzile')       
    # cumulative_market_adj_terzile.to_excel(writer,    sheet_name='market_adj_terzile')
    perf_table.to_excel(writer, sheet_name='Portfolio_Performance')

    ts_absolute.to_excel(writer,  sheet_name='Emissions_Absolute')
    ts_intensity.to_excel(writer, sheet_name='Emissions_Intensity')
    ts_changes.to_excel(writer,   sheet_name='Emissions_Changes')

    cumulative_industry.to_excel(writer,    sheet_name='Industry_Cumulative_Returns')
    perf_industry_df.to_excel(writer,       sheet_name='Industry_Performance')

    # corr_table.to_excel(writer,             sheet_name='Factor_Correlations')

    bmg_stats_df.to_excel(writer,           sheet_name='BMG_Descriptive')
    bmg_corr.to_excel(writer,               sheet_name='BMG_Correlations')
    bmg_cumulative.to_excel(writer,         sheet_name='BMG_Cumulative_Returns')

    table_quintile.to_excel(writer,         sheet_name='BMG_Quintile_ew')
    table_terzile.to_excel(writer,          sheet_name='BMG_Terzile_ew')
    table_industry.to_excel(writer,         sheet_name='BMG_Industry_ew')

    table_vw_quintile.to_excel(writer,         sheet_name='BMG_Quintile_vw')
    table_vw_terzile.to_excel(writer,          sheet_name='BMG_Terzile_vw')
    table_vw_industry.to_excel(writer,         sheet_name='BMG_Industry_vw')

    table_ewe_quintile.to_excel(writer,         sheet_name='BMG_Quintile_ewe')
    table_ewe_terzile.to_excel(writer,          sheet_name='BMG_Terzile_ewe')
    table_ewe_industry.to_excel(writer,         sheet_name='BMG_Industry_ewe')

    table_co2_quintile.to_excel(writer,      sheet_name='CO2_Quintile_ew')
    table_co2_terzile.to_excel(writer,       sheet_name='CO2_Terzile_ew')
    table_co2_industry.to_excel(writer,      sheet_name='CO2_Industry_ew')

    table_vw_co2_quintile.to_excel(writer,      sheet_name='CO2_Quintile_vw')
    table_vw_co2_terzile.to_excel(writer,       sheet_name='CO2_Terzile_vw')
    table_vw_co2_industry.to_excel(writer,      sheet_name='CO2_Industry_vw')

    table_ewe_co2_quintile.to_excel(writer,      sheet_name='CO2_Quintile_ewe')
    table_ewe_co2_terzile.to_excel(writer,       sheet_name='CO2_Terzile_ewe')
    table_ewe_co2_industry.to_excel(writer,      sheet_name='CO2_Industry_ewe')

    # table_full_quintile.to_excel(writer,     sheet_name='Full_Quintile_Regressions')
    # table_full_terzile.to_excel(writer,      sheet_name='Full_Terzile_Regressions')
    # table_full_industry.to_excel(writer,     sheet_name='Full_Industry_Regressions')

    # table_vw_full_quintile.to_excel(writer,     sheet_name='Full_Quintile_vw_Regressions')
    # table_vw_full_terzile.to_excel(writer,      sheet_name='Full_Terzile_vw_Regressions')
    # table_vw_full_industry.to_excel(writer,     sheet_name='Full_Industry_vw_Regressions')

    # table_ewe_full_quintile.to_excel(writer,     sheet_name='Full_Quintile_ewe_Regressions')
    # table_ewe_full_terzile.to_excel(writer,      sheet_name='Full_Terzile_ewe_Regressions')
    # table_ewe_full_industry.to_excel(writer,     sheet_name='Full_Industry_ewe_Regressions')

    df_co2_table.to_excel(writer,   sheet_name='CO2_FF_Regression')

    char_table.to_excel(writer,    sheet_name='BMG_Characteristics')
    spanning_all.to_excel(writer,  sheet_name='Spanning_Regressions')
    bmg_cum_ts.to_excel(writer,    sheet_name='BMG_Cumulative_Comparison')

    # weighting_comparison.to_excel(writer,  sheet_name='Weighting_Comparison')
    # cum_comparison.to_excel(writer,        sheet_name='Weighting_Cum_Returns')

    bmg_all_corr.to_excel(writer, sheet_name='BMG_Factor_Correlations')

    table_dual_quintile.to_excel(writer,  sheet_name='DualBMG_Quintile')
    table_dual_terzile.to_excel(writer,   sheet_name='DualBMG_Terzile')
    table_dual_industry.to_excel(writer,  sheet_name='DualBMG_Industry')

    # table_triple_quintile.to_excel(writer,  sheet_name='TripleBMG_Quintile')
    # table_triple_terzile.to_excel(writer,   sheet_name='TripleBMG_Terzile')
    # table_triple_industry.to_excel(writer,  sheet_name='TripleBMG_Industry')

    table_sub_quintile.to_excel(writer,  sheet_name='Sub2021_Quintile_VW')
    table_sub_terzile.to_excel(writer,   sheet_name='Sub2021_Terzile_VW')
    table_sub_industry.to_excel(writer,  sheet_name='Sub2021_Industry_VW')

    table_vw_full_changes_quintile.to_excel(writer,  sheet_name='VW_BMG_CO2_CH_Quintile')
    table_vw_full_changes_terzile.to_excel(writer,   sheet_name='VW_BMG_CO2_CH_Terzile')
    table_vw_full_changes_industry.to_excel(writer,  sheet_name='VW_BMG_CO2_CH_Industry')

    table_ew_full_changes_quintile.to_excel(writer,  sheet_name='EW_BMG_CO2_CH_Quintile')
    table_ew_full_changes_terzile.to_excel(writer,   sheet_name='EW_BMG_CO2_CH_Terzile')
    table_ew_full_changes_industry.to_excel(writer,  sheet_name='EW_BMG_CO2_CH_Industry')

    table_ewe_full_changes_quintile.to_excel(writer,  sheet_name='EWE_BMG_CO2_CH_Quintile')
    table_ewe_full_changes_terzile.to_excel(writer,   sheet_name='EWE_BMG_CO2_CH_Terzile')
    table_ewe_full_changes_industry.to_excel(writer,  sheet_name='EWE_BMG_CO2_CH_Industry')

    table_chg_vw_quintile.to_excel(writer,  sheet_name='Changes_Quintile_VW')
    table_chg_vw_terzile.to_excel(writer,   sheet_name='Changes_Terzile_VW')
    table_chg_vw_industry.to_excel(writer,  sheet_name='Changes_Industry_VW')

    table_chg_ew_quintile.to_excel(writer,  sheet_name='Changes_Quintile_EW')
    table_chg_ew_terzile.to_excel(writer,   sheet_name='Changes_Terzile_EW')
    table_chg_ew_industry.to_excel(writer,  sheet_name='Changes_Industry_EW')

    table_chg_ewe_quintile.to_excel(writer,  sheet_name='Changes_Quintile_EWE')
    table_chg_ewe_terzile.to_excel(writer,   sheet_name='Changes_Terzile_EWE')
    table_chg_ewe_industry.to_excel(writer,  sheet_name='Changes_Industry_EWE')